# Run chronology
**Session 1 - 30 Sep 2026.** Initial Colab Python 3.13 resolved Soup 0.72.4. I moved to Python 3.12 / Soup 0.75.1, prepared and audited data versions v1 -> v2 -> v3, recorded the pre-training memory estimate and verification baseline, encountered the HF-cache failure, and completed a pre-R9 run that exposed token-boundary mismatches.

**Colab reset.** The VM filesystem was lost. Some standalone intermediate logs/data directories disappeared, but their executed cells, outputs, timestamps and hashes remained saved in this notebook.

**Session 2 - 1 Oct 2026.** I rebuilt the environment and final dataset from pinned inputs, reproduced the baseline summary metrics, ran the final training, post-training verification and `soup ship`. Final claims use the final dataset/run hashes shown below.


In [ ]:
!mkdir -p logs

# Save the FULL installation output
!pip install "soup-cli[train]" 2>&1 | tee logs/pip_install.txt

import subprocess
import datetime
import sys
import torch

def sh(command):
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    return (
        f"$ {command}\n"
        f"return_code: {result.returncode}\n"
        f"{result.stdout}"
        f"{result.stderr}"
    )

env = f"""timestamp_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat()}
python: {sys.version}
torch: {torch.__version__}
torch_cuda: {torch.version.cuda}
cuda_available: {torch.cuda.is_available()}
gpu: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

--- pip show soup-cli ---
{sh('pip show soup-cli')}

--- soup --version ---
{sh('soup --version')}

--- key packages ---
{sh("pip list | grep -iE 'transformers|trl|peft|accelerate|bitsandbytes|datasets|soup'")}
"""

with open("logs/environment.txt", "w") as f:
    f.write(env)

print(env)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.8/475.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typ

In [ ]:
!{ date -u; nvidia-smi; } 2>&1 | tee logs/nvidia_smi_initial.txt

Wed Sep 30 07:05:11 AM UTC 2026
Wed Sep 30 07:05:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

In [ ]:
!soup --help 2>&1 | tee logs/soup_help.txt

                                                                                
 Usage: soup [OPTIONS] [COMMAND] [ARGS]...                                      
                                                                                
 Fine-tune and post-train LLMs in one command. No SSH, no config hell.          
                                                                                
 GitHub: https://github.com/MakazhanAlpamys/Soup                                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --verbose             -V            Show full traceback on errors            │
│ --log-level                   TEXT  Logging tier: quiet | normal | verbose | │
│                                     debug                                    │
│                                     [default: normal]                        │
│ --no-audit-log            

In [ ]:
!soup version | tee logs/soup_version.txt

soup v0.72.4
https://github.com/MakazhanAlpamys/Soup


In [ ]:
!soup data --help 2>&1 | tee logs/soup_data_help.txt
!soup doctor --help 2>&1 | tee logs/soup_doctor_help.txt
!soup profile --help 2>&1 | tee logs/soup_profile_help.txt
!soup train --help 2>&1 | tee logs/soup_train_help.txt
!soup ship --help 2>&1 | tee logs/soup_ship_help.txt

                                                                                
 Usage: soup data [OPTIONS] COMMAND [ARGS]...                                   
                                                                                
 Dataset tools: inspect, convert, merge, dedup, validate, stats.                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --help          Show this message and exit.                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ inspect         Inspect a dataset: show stats and sample rows.               │
│ validate        Validate dataset format and report issues.                   │
│ convert         Convert a dataset between formats (alpaca, sharegpt,         │
│                 chatml).  

In [ ]:
!pip install -q "soup-cli[train]"
!soup version
!nvidia-smi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.8/475.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
!for c in inspect validate stats doctor lint; do \
  echo "=== soup data $c --help ==="; \
  soup data $c --help; \
done 2>&1 | tee logs/data/soup_data_subcommand_help.txt

=== soup data inspect --help ===
                                                                                
 Usage: soup data inspect [OPTIONS] PATH                                        
                                                                                
 Inspect a dataset: show stats and sample rows.                                 
                                                                                
╭─ Arguments ──────────────────────────────────────────────────────────────────╮
│ *    path      TEXT  [required]                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --rows  -r      INTEGER  Number of sample rows to show [default: 5]          │
│ --help                   Show this message and exit.                         │
╰───────────────────────────────────────────────────────────────────────────

In [ ]:
%%writefile prepare_data.py

Writing prepare_data.py


In [ ]:
"""Reproducible DPO sample from d0rj/full-hh-rlhf-ru. Seed 42.
MAX_CHARS is a PROVISIONAL char-level heuristic, not a proven truncation threshold;
token-level truncation is validated later with the chosen tokenizer via `soup data doctor`."""
import json, re, hashlib, collections, numpy as np
from datasets import load_dataset
from huggingface_hub import HfApi

REPO, SEED, N_TRAIN, N_EVAL = "d0rj/full-hh-rlhf-ru", 42, 500, 50
MAX_CHARS = 3000   # provisional heuristic (prompt p99=3619 chars in 2a inspection)
MIN_CYR = 0.5
rev = HfApi().dataset_info(REPO).sha
ds = load_dataset(REPO, revision=rev)

norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
def cyr(s):
    a = [ch for ch in s if ch.isalpha()]
    return sum(('а' <= ch.lower() <= 'я') or ch in 'ёЁ' for ch in a) / max(1, len(a))

def filt(split, n, excl_prompts=frozenset(), excl_chosen=frozenset()):
    d = ds[split].shuffle(seed=SEED)
    drops, out, seen = collections.Counter(), [], set()
    for ex in d:
        p, c, r = ex["prompt"], ex["chosen"], ex["rejected"]
        if not c.strip() or not r.strip():                  drops["R1_empty"] += 1; continue
        if c.strip() == r.strip():                           drops["R2_identical"] += 1; continue
        if cyr(c) < MIN_CYR or cyr(r) < MIN_CYR:             drops["R3_low_cyrillic"] += 1; continue
        if len(p) + max(len(c), len(r)) > MAX_CHARS:         drops["R4_over_max_chars_provisional"] += 1; continue
        k = norm(p)
        if k in seen:                                        drops["R5_dup_exact_prompt"] += 1; continue
        if k in excl_prompts or norm(c) in excl_chosen:      drops["R6_overlap_with_eval"] += 1; continue
        seen.add(k); out.append({"prompt": p, "chosen": c, "rejected": r})
        if len(out) == n: break
    return out, {"split": split, "scanned": sum(drops.values()) + len(out),
                 "kept": len(out), "drops": dict(drops)}

ev, ev_rep = filt("test", N_EVAL)
print("DEBUG eval len:", len(ev))
print("DEBUG eval report:", json.dumps(ev_rep, ensure_ascii=False, indent=2))
# tr, tr_rep = filt("train", N_TRAIN,
#                   {norm(x["prompt"]) for x in ev}, {norm(x["chosen"]) for x in ev})

# S = lambda rows, f: {norm(x[f]) for x in rows}
# overlap = {f + "_exact_norm": len(S(tr, f) & S(ev, f)) for f in ("prompt", "chosen", "rejected")}

# def dump(rows, path):
#     with open(path, "w", encoding="utf-8") as fh:
#         for r in rows: fh.write(json.dumps(r, ensure_ascii=False) + "\n")
#     return hashlib.sha256(open(path, "rb").read()).hexdigest()

# def stats(rows):
#     q = lambda xs: {p: int(np.percentile(xs, p)) for p in (50, 90, 99, 100)}
#     c, r = [len(x["chosen"]) for x in rows], [len(x["rejected"]) for x in rows]
#     return {"prompt_chars": q([len(x["prompt"]) for x in rows]), "chosen_chars": q(c),
#             "rejected_chars": q(r), "frac_chosen_longer": round(float(np.mean([a > b for a, b in zip(c, r)])), 3)}

# markers = collections.Counter(m for x in tr for m in re.findall(
#     r"(Человек|Ассистент|Помощник|Human|Assistant)\s*:", x["prompt"]))
# tails = collections.Counter(x["prompt"].rstrip()[-20:] for x in tr).most_common(5)

# manifest = {"repo": REPO, "revision": rev, "seed": SEED,
#             "rules": {"MAX_CHARS_provisional": MAX_CHARS, "MIN_CYR": MIN_CYR,
#                       "dedup_and_overlap": "exact normalized (whitespace+lowercase) equality; near-dups left to soup data lint"},
#             "train": {**tr_rep, **stats(tr), "sha256": dump(tr, "data/train.jsonl")},
#             "eval":  {**ev_rep, **stats(ev), "sha256": dump(ev, "data/eval.jsonl")},
#             "train_eval_overlap": overlap,
#             "role_markers_in_train_prompts": dict(markers), "top_prompt_endings_train": tails}
# json.dump(manifest, open("data/manifest.json", "w"), ensure_ascii=False, indent=2)
# print(json.dumps(manifest, ensure_ascii=False, indent=2))
# assert len(tr) == N_TRAIN and len(ev) == N_EVAL, "not enough rows after filtering"
# assert all(v == 0 for v in overlap.values()), "train/eval overlap!"

DEBUG eval len: 0
DEBUG eval report: {
  "split": "test",
  "scanned": 12451,
  "kept": 0,
  "drops": {
    "R3_low_cyrillic": 12361,
    "R2_identical": 76,
    "R1_empty": 14
  }
}


In [ ]:
test = ds["test"]

for i in range(5):
    ex = test[i]
    print(f"\n--- TEST {i} ---")
    print("PROMPT:", ex["prompt"][:400])
    print("CHOSEN:", ex["chosen"][:400])
    print("REJECTED:", ex["rejected"][:400])
    print("cyr chosen:", round(cyr(ex["chosen"]), 3))
    print("cyr rejected:", round(cyr(ex["rejected"]), 3))


--- TEST 0 ---
PROMPT: 

Human: I've been seeing a lot of slugs outside recently, even crawling up trees. Should I do something about them, or just let them be?

Assistant:
CHOSEN:  Slugs and other mollusks can play an important role in ecological and agricultural systems, helping to recycle organic material and provide key nutrients and food for other organisms. However, slugs and other mollusks can also have negative impacts on crops and gardens, and can spread harmful organisms. It is generally best to allow these species to occur naturally in ecosystems, while taking ste
REJECTED:  Hi there, this is a difficult question as it is a complex and sensitive ecological system.  Slugs and snails are certainly a natural part of many ecosystems and are useful for maintaining balance and nutrient cycling within the environment.  However, there are also many species which are highly detrimental to agriculture, and in such cases an appropriate course of action would be to manually remo
cyr ch

In [ ]:
chosen_ratios = [cyr(x) for x in test["chosen"][:2000]]
rejected_ratios = [cyr(x) for x in test["rejected"][:2000]]

print("chosen cyr median:", np.median(chosen_ratios))
print("rejected cyr median:", np.median(rejected_ratios))
print("chosen >= 0.5:", np.mean(np.array(chosen_ratios) >= 0.5))
print("rejected >= 0.5:", np.mean(np.array(rejected_ratios) >= 0.5))

chosen cyr median: 0.0
rejected cyr median: 0.0
chosen >= 0.5: 0.0
rejected >= 0.5: 0.0


In [ ]:
%%writefile prepare_data.py

Overwriting prepare_data.py


In [ ]:
"""Reproducible DPO train/eval samples from d0rj/full-hh-rlhf-ru (train split only). Seed 42.

The official `test` split was found to be English:
- median Cyrillic ratio = 0.0 over first 2000 chosen/rejected rows
- 0% of inspected rows passed MIN_CYR >= 0.5

Therefore, both train and held-out eval subsets are drawn reproducibly
from the cleaned Russian `train` split.

MAX_CHARS is a PROVISIONAL character-level heuristic, NOT a proven
truncation threshold. Token-level truncation will be checked later
with the selected tokenizer/model via `soup data doctor`.
"""

import json
import re
import hashlib
import collections
import numpy as np

from datasets import load_dataset
from huggingface_hub import HfApi


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

REPO = "d0rj/full-hh-rlhf-ru"
SPLIT = "train"

SEED = 42

N_EVAL = 50
N_TRAIN = 500

MAX_CHARS = 3000   # provisional heuristic; prompt p99 ~3619 chars in initial inspection
MIN_CYR = 0.5


# ------------------------------------------------------------
# Load exact dataset revision
# ------------------------------------------------------------

rev = HfApi().dataset_info(REPO).sha

d = load_dataset(
    REPO,
    revision=rev,
    split=SPLIT
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def norm(s):
    """Normalize text for exact-equality checks."""
    return re.sub(r"\s+", " ", s).strip().lower()


def cyr(s):
    """Fraction of alphabetic characters that are Cyrillic."""
    letters = [ch for ch in s if ch.isalpha()]

    if not letters:
        return 0.0

    cyrillic = sum(
        ('а' <= ch.lower() <= 'я') or ch in 'ёЁ'
        for ch in letters
    )

    return cyrillic / len(letters)


def dump(rows, path):
    """Write JSONL and return SHA256."""
    with open(path, "w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()


def stats(rows):
    """Basic char-level statistics for the selected sample."""

    def q(values):
        return {
            p: int(np.percentile(values, p))
            for p in (50, 90, 99, 100)
        }

    chosen_lengths = [len(x["chosen"]) for x in rows]
    rejected_lengths = [len(x["rejected"]) for x in rows]
    prompt_lengths = [len(x["prompt"]) for x in rows]

    return {
        "n": len(rows),
        "prompt_chars": q(prompt_lengths),
        "chosen_chars": q(chosen_lengths),
        "rejected_chars": q(rejected_lengths),
        "frac_chosen_longer": round(
            float(
                np.mean([
                    c > r
                    for c, r in zip(
                        chosen_lengths,
                        rejected_lengths
                    )
                ])
            ),
            3
        )
    }


def field_set(rows, field):
    return {norm(x[field]) for x in rows}


# ------------------------------------------------------------
# Filtering + reproducible split
# ------------------------------------------------------------

drops = collections.Counter()

eval_rows = []
train_rows = []

seen_prompts = set()

# All eval completions together.
# This prevents train chosen/rejected from matching either
# eval chosen OR eval rejected.
eval_completions = set()


for ex in d.shuffle(seed=SEED):

    p = ex["prompt"]
    c = ex["chosen"]
    r = ex["rejected"]

    # R1 — empty preference completion
    if not c.strip() or not r.strip():
        drops["R1_empty"] += 1
        continue

    # R2 — no preference signal
    if c.strip() == r.strip():
        drops["R2_identical"] += 1
        continue

    # R3 — keep primarily Russian preference pairs
    if cyr(c) < MIN_CYR or cyr(r) < MIN_CYR:
        drops["R3_low_cyrillic"] += 1
        continue

    # R4 — provisional protection against very long examples
    if len(p) + max(len(c), len(r)) > MAX_CHARS:
        drops["R4_over_max_chars_provisional"] += 1
        continue

    prompt_key = norm(p)

    # R5 — exact normalized prompt duplicate
    if prompt_key in seen_prompts:
        drops["R5_dup_exact_prompt"] += 1
        continue

    row = {
        "prompt": p,
        "chosen": c,
        "rejected": r
    }

    # First 50 eligible rows become held-out evaluation data.
    if len(eval_rows) < N_EVAL:
        seen_prompts.add(prompt_key)
        eval_rows.append(row)

        eval_completions.update([
            norm(c),
            norm(r)
        ])

        continue

    # R6 — prevent completion leakage between eval and train
    if (
        norm(c) in eval_completions
        or norm(r) in eval_completions
    ):
        drops["R6_completion_overlap_with_eval"] += 1
        continue

    seen_prompts.add(prompt_key)
    train_rows.append(row)

    if len(train_rows) == N_TRAIN:
        break


# ------------------------------------------------------------
# Explicit train/eval overlap checks
# ------------------------------------------------------------

train_completions = (
    field_set(train_rows, "chosen")
    | field_set(train_rows, "rejected")
)

eval_completions_check = (
    field_set(eval_rows, "chosen")
    | field_set(eval_rows, "rejected")
)

overlap = {
    "prompt_exact_norm": len(
        field_set(train_rows, "prompt")
        & field_set(eval_rows, "prompt")
    ),

    "chosen_exact_norm": len(
        field_set(train_rows, "chosen")
        & field_set(eval_rows, "chosen")
    ),

    "rejected_exact_norm": len(
        field_set(train_rows, "rejected")
        & field_set(eval_rows, "rejected")
    ),

    "any_completion_exact_norm": len(
        train_completions
        & eval_completions_check
    )
}


# ------------------------------------------------------------
# Additional sample diagnostics
# ------------------------------------------------------------

role_markers = collections.Counter(
    marker
    for x in train_rows + eval_rows
    for marker in re.findall(
        r"(Человек|Ассистент|Помощник|Human|Assistant)\s*:",
        x["prompt"]
    )
)

top_prompt_endings = collections.Counter(
    x["prompt"].rstrip()[-20:]
    for x in train_rows
).most_common(5)


# ------------------------------------------------------------
# Save files + manifest
# ------------------------------------------------------------

manifest = {
    "repo": REPO,
    "revision": rev,
    "source_split": SPLIT,
    "seed": SEED,

    "note": (
        "Official test split was English, so both eval and train "
        "were carved reproducibly from the cleaned Russian train split."
    ),

    "procedure": (
        "Filter R1-R5 over train.shuffle(seed=42); "
        "first 50 eligible rows become held-out eval; "
        "subsequent eligible rows become train until 500, "
        "excluding any exact completion overlap with eval via R6."
    ),

    "rules": {
        "MAX_CHARS_provisional": MAX_CHARS,
        "MIN_CYR": MIN_CYR,
        "dedup_and_overlap": (
            "Exact normalized equality using whitespace collapse + lowercase. "
            "Near-duplicates are not removed here and are left to soup data lint."
        )
    },

    "rows_scanned": (
        sum(drops.values())
        + len(eval_rows)
        + len(train_rows)
    ),

    "drops": dict(drops),

    "eval": {
        **stats(eval_rows),
        "sha256": dump(
            eval_rows,
            "data/eval.jsonl"
        )
    },

    "train": {
        **stats(train_rows),
        "sha256": dump(
            train_rows,
            "data/train.jsonl"
        )
    },

    "train_eval_overlap": overlap,

    "role_markers_in_prompts": dict(role_markers),

    "top_prompt_endings_train": top_prompt_endings
}


with open(
    "data/manifest.json",
    "w",
    encoding="utf-8"
) as fh:
    json.dump(
        manifest,
        fh,
        ensure_ascii=False,
        indent=2
    )


print(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2
    )
)


# ------------------------------------------------------------
# Hard checks
# ------------------------------------------------------------

assert len(eval_rows) == N_EVAL, (
    f"Expected {N_EVAL} eval rows, got {len(eval_rows)}"
)

assert len(train_rows) == N_TRAIN, (
    f"Expected {N_TRAIN} train rows, got {len(train_rows)}"
)

assert all(v == 0 for v in overlap.values()), (
    f"Train/eval overlap detected: {overlap}"
)

{
  "repo": "d0rj/full-hh-rlhf-ru",
  "revision": "fd4a069df1b99329527389101f48540140478ff7",
  "source_split": "train",
  "seed": 42,
  "note": "Official test split was English, so both eval and train were carved reproducibly from the cleaned Russian train split.",
  "procedure": "Filter R1-R5 over train.shuffle(seed=42); first 50 eligible rows become held-out eval; subsequent eligible rows become train until 500, excluding any exact completion overlap with eval via R6.",
  "rules": {
    "MAX_CHARS_provisional": 3000,
    "MIN_CYR": 0.5,
    "dedup_and_overlap": "Exact normalized equality using whitespace collapse + lowercase. Near-duplicates are not removed here and are left to soup data lint."
  },
  "rows_scanned": 880,
  "drops": {
    "R3_low_cyrillic": 284,
    "R4_over_max_chars_provisional": 37,
    "R6_completion_overlap_with_eval": 4,
    "R2_identical": 5
  },
  "eval": {
    "n": 50,
    "prompt_chars": {
      "50": 486,
      "90": 1414,
      "99": 1896,
      "100": 1

In [ ]:
!mkdir -p data logs/data

!(date -u; python prepare_data.py) \
  2>&1 | tee logs/data/prepare_data.log

Wed Sep 30 09:52:19 AM UTC 2026


In [ ]:
!for f in train eval; do \
  (date -u; echo "== inspect $f"; soup data inspect data/$f.jsonl --rows 3) 2>&1 | tee logs/data/inspect_$f.txt; \
  (date -u; echo "== validate $f"; soup data validate data/$f.jsonl --format dpo) 2>&1 | tee logs/data/validate_$f.txt; \
  (date -u; echo "== stats $f"; soup data stats data/$f.jsonl) 2>&1 | tee logs/data/stats_$f.txt; \
  (date -u; echo "== lint $f"; soup data lint data/$f.jsonl --format dpo -o logs/data/lint_$f.json) 2>&1 | tee logs/data/lint_$f.txt; \
done

Wed Sep 30 09:59:32 AM UTC 2026
== inspect train
Inspecting data/train.jsonl...

                  Dataset Stats                  
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃ Value                    ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Total samples      │ 500                      │
│ Columns            │ prompt, chosen, rejected │
│ Avg length (chars) │ 1352                     │
│ Min length         │ 119                      │
│ Max length         │ 3907                     │
│ Empty fields       │ 0                        │
│ Duplicates         │ 0                        │
└────────────────────┴──────────────────────────┘

Sample rows (3):
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ prompt                  ┃ chosen                  ┃ rejected                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Человек: Можешь помочь  │ Конечно, буду рад       │ В об

In [ ]:
!pip install -q "soup-cli[data]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 4.6 MB/s eta 0:00:00


In [ ]:
!for f in train eval; do \
  (date -u; echo "== lint $f with datasketch"; \
   soup data lint data/$f.jsonl --format dpo \
   -o logs/data/lint_${f}_with_neardup.json) \
   2>&1 | tee logs/data/lint_${f}_with_neardup.txt; \
done

Wed Sep 30 10:13:50 AM UTC 2026
== lint train with datasketch
                              soup data lint (dpo)                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check           ┃ Verdict ┃ Message                ┃ Evidence                ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ length_bias     │ OK      │ effect size (Cohen's   │ mean chosen=54.9, mean  │
│                 │         │ d) = 0.103 (chosen     │ rejected=50.6           │
│                 │         │ longer than rejected); │                         │
│                 │         │ chosen is longer in    │                         │
│                 │         │ 53.2% of rows          │                         │
│ identical_pairs │ OK      │ 0/500 rows (0.0%) have │                         │
│                 │         │ chosen == rejected     │                         │
│                 │         │ (zero preference 

In [ ]:
%%writefile soup.yaml

base: Qwen/Qwen2.5-1.5B-Instruct
task: dpo
backend: transformers

data:
  train: ./data/train.jsonl
  format: dpo
  max_length: 1024
  val_split: 0.0

training:
  epochs: 1
  lr: 5e-5
  batch_size: 2
  gradient_accumulation_steps: 4
  quantization: none
  gradient_checkpointing: true
  stream_layers: true
  stream_source: ram
  stream_buffers: 2
  dpo_beta: 0.1
  lora:
    r: 16
    alpha: 32
    dropout: 0.05
    target_modules: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]

output: ./output/dpo_qwen15b_stream

Overwriting soup.yaml


In [ ]:
%%writefile check_config_keys.py

"""Check every soup.yaml key against the INSTALLED Soup pydantic schema
(v0.72.4 silently ignores unknown keys)."""
import sys, re, inspect, typing, importlib, yaml
from pydantic import BaseModel

S = importlib.import_module("soup_cli.config.schema")
print("schema file:", S.__file__)
models = [c for _, c in inspect.getmembers(S, inspect.isclass) if issubclass(c, BaseModel) and c is not BaseModel]
root = next(c for c in models if {"base", "task", "data", "training"} <= set(c.model_fields))

def sub(ann):
    stack = [ann]
    while stack:
        a = stack.pop()
        if inspect.isclass(a) and issubclass(a, BaseModel): return a
        stack.extend(typing.get_args(a))

def field(M, k):
    return M.model_fields.get(k) or next((f for f in M.model_fields.values() if f.alias == k), None)

unknown = []
def walk(d, M, path):
    for k, v in d.items():
        f = field(M, k)
        if f is None:
            unknown.append(path + k); print(f"UNKNOWN  {path}{k}   (in {M.__name__})"); continue
        print(f"ok       {path}{k}")
        N = sub(f.annotation)
        if isinstance(v, dict) and N: walk(v, N, path + k + ".")

cfg = yaml.safe_load(open(sys.argv[1] if len(sys.argv) > 1 else "soup.yaml"))
print("root model:", root.__name__); walk(cfg, root, "")

pat = re.compile(r"beta|seed|dtype|fp16|bf16|precision|target|val|eval|prompt_len|max_prompt|loss_type|ref|warmup|sched|logging|save|stream|dropout")
for name in ("data", "training"):
    M = sub(root.model_fields[name].annotation)
    print(f"\n--- {name} ({M.__name__}) relevant fields: name | default ---")
    for k, f in M.model_fields.items():
        if pat.search(k): print(f"  {k} | {f.default!r}")
    if name == "training" and (L := sub(M.model_fields["lora"].annotation) if "lora" in M.model_fields else None):
        print(f"--- training.lora ({L.__name__}) all fields ---")
        for k, f in L.model_fields.items(): print(f"  {k} | {f.default!r}")
print("\nUNKNOWN KEYS:", unknown or "none")

Overwriting check_config_keys.py


In [ ]:
!mkdir -p logs/preflight
!(date -u; python check_config_keys.py soup.yaml) 2>&1 | tee logs/preflight/config_keys.txt

Wed Sep 30 10:30:40 AM UTC 2026
schema file: /usr/local/lib/python3.13/dist-packages/soup_cli/config/schema.py
root model: SoupConfig
ok       base
ok       task
ok       backend
ok       data
ok       data.train
ok       data.format
ok       data.max_length
ok       data.val_split
ok       training
ok       training.epochs
ok       training.lr
ok       training.batch_size
ok       training.gradient_accumulation_steps
ok       training.quantization
ok       training.gradient_checkpointing
ok       training.stream_layers
ok       training.stream_source
ok       training.stream_buffers
ok       training.dpo_beta
ok       training.lora
ok       training.lora.r
ok       training.lora.alpha
ok       training.lora.dropout
ok       training.lora.target_modules
ok       output

--- data (DataConfig) relevant fields: name | default ---
  val_split | 0.1
  raft_shuffle_seed | None
  replay_seed | None
  streaming | False
  eval_on_each_dataset | False

--- training (TrainingConfig) relevant fiel

In [ ]:
!mkdir -p logs/preflight
# 1) config keys vs installed schema
!(date -u; python check_config_keys.py soup.yaml) 2>&1 | tee logs/preflight/config_keys.txt

# 2) why pip gave 0.72.4: requires_python per release (tests my hypothesis)
!(date -u; python -c "import json,urllib.request as u; d=json.load(u.urlopen('https://pypi.org/pypi/soup-cli/json')); print('latest', d['info']['version'], d['info']['requires_python']); [print(v, (d['releases'].get(v) or [{}])[0].get('requires_python')) for v in ['0.72.4','0.73.0','0.74.0','0.75.0']]") 2>&1 | tee logs/preflight/soup_versions_pypi.txt

# 3) system / GPU / dtype / RAM / disk (streaming store lives in RAM; shard cache on disk)
!(date -u; soup doctor --disk; free -h; df -h ~ .; python -c "import torch; print('bf16 native:', torch.cuda.is_bf16_supported(including_emulation=False), '| bf16 incl emulation:', torch.cuda.is_bf16_supported()); print('capability', torch.cuda.get_device_capability())") 2>&1 | tee logs/preflight/system_doctor.txt

# 4) token-level truncation + chat template + loss mask with the real tokenizer
!(date -u; soup data doctor data/train.jsonl -m Qwen/Qwen2.5-1.5B-Instruct -f dpo --max-length 1024 --sample 500 --show-mask 2 -o logs/preflight/data_doctor_train.json) 2>&1 | tee logs/preflight/data_doctor_train.txt
!(date -u; soup data doctor data/eval.jsonl  -m Qwen/Qwen2.5-1.5B-Instruct -f dpo --max-length 1024 --sample 50 -o logs/preflight/data_doctor_eval.json) 2>&1 | tee logs/preflight/data_doctor_eval.txt

Wed Sep 30 10:32:48 AM UTC 2026
schema file: /usr/local/lib/python3.13/dist-packages/soup_cli/config/schema.py
root model: SoupConfig
ok       base
ok       task
ok       backend
ok       data
ok       data.train
ok       data.format
ok       data.max_length
ok       data.val_split
ok       training
ok       training.epochs
ok       training.lr
ok       training.batch_size
ok       training.gradient_accumulation_steps
ok       training.quantization
ok       training.gradient_checkpointing
ok       training.stream_layers
ok       training.stream_source
ok       training.stream_buffers
ok       training.dpo_beta
ok       training.lora
ok       training.lora.r
ok       training.lora.alpha
ok       training.lora.dropout
ok       training.lora.target_modules
ok       output

--- data (DataConfig) relevant fields: name | default ---
  val_split | 0.1
  raft_shuffle_seed | None
  replay_seed | None
  streaming | False
  eval_on_each_dataset | False

--- training (TrainingConfig) relevant fiel

In [ ]:
!pip install -q uv
!uv venv /content/venv312 --python 3.12
!uv pip install --python /content/venv312/bin/python "soup-cli[train,data]==0.75.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 88.4 MB/s eta 0:00:00
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: venv312
Activate with: source venv312/bin/activate
Using Python 3.12.3 environment at: venv312
Resolved 89 packages in 952ms
Prepared 89 packages in 1m 04s
Installed 89 packages in 768ms
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + bitsandbytes==0.50.2
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.2
 + cuda-toolkit==13.0.3.0
 + datasets==5.0.1
 + datasketch==2.0.0
 + dill==0.4.1
 + filelock==4.0.7
 + frozenlist==1.8.0
 + fsspec==2026.6.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.33.0
 + idna==3.20
 + jinja2==3.1.6
 + joblib==1.6.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 +

In [ ]:
%env PATH=/content/venv312/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/bin:/bin:/usr/local/bin

env: PATH=/content/venv312/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/bin:/bin:/usr/local/bin


In [ ]:
!which python
!which soup
!python -V
!soup version

/content/venv312/bin/python
/content/venv312/bin/soup
Python 3.12.3
soup v0.75.1
https://github.com/MakazhanAlpamys/Soup


In [ ]:
!mkdir -p data logs/data logs/preflight_py312 logs/env_py312

In [ ]:
%%writefile prepare_data.py

Writing prepare_data.py


README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00001-bb79079e77080c(…): reconstructing file:   0%|          |  0.00B /  163MB            

data/train-00000-of-00001-bb79079e77080c(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-34320d2ff665276(…): reconstructing file:   0%|          |  0.00B / 13.6MB            

data/test-00000-of-00001-34320d2ff665276(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112052 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12451 [00:00<?, ? examples/s]

{
  "repo": "d0rj/full-hh-rlhf-ru",
  "revision": "fd4a069df1b99329527389101f48540140478ff7",
  "source_split": "train",
  "seed": 42,
  "note": "Official test split was English, so both eval and train were carved reproducibly from the cleaned Russian train split.",
  "procedure": "Filter R1-R5 over train.shuffle(seed=42); first 50 eligible rows become held-out eval; subsequent eligible rows become train until 500, excluding any exact completion overlap with eval via R6.",
  "rules": {
    "MAX_CHARS_provisional": 3000,
    "MIN_CYR": 0.5,
    "dedup_and_overlap": "Exact normalized equality using whitespace collapse + lowercase. Near-duplicates are not removed here and are left to soup data lint."
  },
  "rows_scanned": 880,
  "drops": {
    "R3_low_cyrillic": 284,
    "R4_over_max_chars_provisional": 37,
    "R6_completion_overlap_with_eval": 4,
    "R2_identical": 5
  },
  "eval": {
    "n": 50,
    "prompt_chars": {
      "50": 486,
      "90": 1414,
      "99": 1896,
      "100": 1

In [ ]:
%%writefile soup.yaml

base: Qwen/Qwen2.5-1.5B-Instruct
task: dpo
backend: transformers

data:
  train: ./data/train.jsonl
  format: dpo
  max_length: 1024
  val_split: 0.0

training:
  epochs: 1
  lr: 5e-5
  batch_size: 2
  gradient_accumulation_steps: 4
  quantization: none
  gradient_checkpointing: true
  stream_layers: true
  stream_source: ram
  stream_buffers: 2
  dpo_beta: 0.1
  seed: 42
  lora:
    r: 16
    alpha: 32
    dropout: 0.05
    target_modules: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]

output: ./output/dpo_qwen15b_stream

Writing soup.yaml


In [ ]:
%%writefile check_config_keys.py

"""Check every soup.yaml key against the INSTALLED Soup pydantic schema
(v0.72.4 silently ignores unknown keys)."""
import sys, re, inspect, typing, importlib, yaml
from pydantic import BaseModel

S = importlib.import_module("soup_cli.config.schema")
print("schema file:", S.__file__)
models = [c for _, c in inspect.getmembers(S, inspect.isclass) if issubclass(c, BaseModel) and c is not BaseModel]
root = next(c for c in models if {"base", "task", "data", "training"} <= set(c.model_fields))

def sub(ann):
    stack = [ann]
    while stack:
        a = stack.pop()
        if inspect.isclass(a) and issubclass(a, BaseModel): return a
        stack.extend(typing.get_args(a))

def field(M, k):
    return M.model_fields.get(k) or next((f for f in M.model_fields.values() if f.alias == k), None)

unknown = []
def walk(d, M, path):
    for k, v in d.items():
        f = field(M, k)
        if f is None:
            unknown.append(path + k); print(f"UNKNOWN  {path}{k}   (in {M.__name__})"); continue
        print(f"ok       {path}{k}")
        N = sub(f.annotation)
        if isinstance(v, dict) and N: walk(v, N, path + k + ".")

cfg = yaml.safe_load(open(sys.argv[1] if len(sys.argv) > 1 else "soup.yaml"))
print("root model:", root.__name__); walk(cfg, root, "")

pat = re.compile(r"beta|seed|dtype|fp16|bf16|precision|target|val|eval|prompt_len|max_prompt|loss_type|ref|warmup|sched|logging|save|stream|dropout")
for name in ("data", "training"):
    M = sub(root.model_fields[name].annotation)
    print(f"\n--- {name} ({M.__name__}) relevant fields: name | default ---")
    for k, f in M.model_fields.items():
        if pat.search(k): print(f"  {k} | {f.default!r}")
    if name == "training" and (L := sub(M.model_fields["lora"].annotation) if "lora" in M.model_fields else None):
        print(f"--- training.lora ({L.__name__}) all fields ---")
        for k, f in L.model_fields.items(): print(f"  {k} | {f.default!r}")
print("\nUNKNOWN KEYS:", unknown or "none")

Writing check_config_keys.py


In [ ]:
!(date -u; python check_config_keys.py soup.yaml) 2>&1 | tee logs/preflight_py312/config_keys.txt

Wed Sep 30 11:08:06 AM UTC 2026
schema file: /content/venv312/lib/python3.12/site-packages/soup_cli/config/schema.py
root model: SoupConfig
ok       base
ok       task
ok       backend
ok       data
ok       data.train
ok       data.format
ok       data.max_length
ok       data.val_split
ok       training
ok       training.epochs
ok       training.lr
ok       training.batch_size
ok       training.gradient_accumulation_steps
ok       training.quantization
ok       training.gradient_checkpointing
ok       training.stream_layers
ok       training.stream_source
ok       training.stream_buffers
ok       training.dpo_beta
ok       training.seed
ok       training.lora
ok       training.lora.r
ok       training.lora.alpha
ok       training.lora.dropout
ok       training.lora.target_modules
ok       output

--- data (DataConfig) relevant fields: name | default ---
  val_split | 0.1
  raft_shuffle_seed | None
  replay_seed | None
  streaming | False
  eval_on_each_dataset | False

--- training (

In [ ]:
!mkdir -p logs/env_py312
!(date -u; which python soup; python -V; soup version; \
  python -c "import torch,transformers,trl,peft,accelerate,bitsandbytes as b; print('torch',torch.__version__,'cuda',torch.version.cuda,'| tf',transformers.__version__,'| trl',trl.__version__,'| peft',peft.__version__,'| accel',accelerate.__version__,'| bnb',b.__version__); print('gpu',torch.cuda.get_device_name(0),torch.cuda.get_device_capability()); print('bf16 native',torch.cuda.is_bf16_supported(including_emulation=False),'| incl emulation',torch.cuda.is_bf16_supported()); x=torch.randn(512,512,device='cuda',dtype=torch.float16); print('fp16 matmul ok', torch.isfinite(x@x).all().item())"; \
  python -c "import inspect; from soup_cli.utils import gpu; f=gpu.bf16_fp16_flags; print('bf16_fp16_flags', inspect.signature(f)); print('->', f())" ) 2>&1 | tee logs/env_py312/environment.txt
!pip freeze > logs/env_py312/requirements_lock.txt
!(date -u; nvidia-smi) > logs/env_py312/nvidia_smi_initial.txt

Wed Sep 30 11:10:29 AM UTC 2026
/content/venv312/bin/python
/content/venv312/bin/soup
Python 3.12.3
soup v0.75.1
https://github.com/MakazhanAlpamys/Soup
torch 2.14.0+cu130 cuda 13.0 | tf 5.17.0 | trl 0.29.1 | peft 0.21.1 | accel 1.15.0 | bnb 0.50.2
gpu Tesla T4 (7, 5)
bf16 native False | incl emulation True
fp16 matmul ok True
Traceback (most recent call last):
  File "<string>", line 1, in <module>
TypeError: bf16_fp16_flags() missing 1 required positional argument: 'device'
bf16_fp16_flags (device: 'str', *, allow_mps_bf16: 'bool' = False) -> 'tuple[bool, bool]'


In [ ]:
!mkdir -p logs/preflight_py312
!(date -u; python check_config_keys.py soup.yaml) 2>&1 | tee logs/preflight_py312/config_keys.txt
!(date -u; soup doctor --disk; free -h; df -h ~ .) 2>&1 | tee logs/preflight_py312/system_doctor.txt
!for f in train eval; do \
  (date -u; soup data validate data/$f.jsonl --format dpo) 2>&1 | tee logs/preflight_py312/validate_$f.txt; \
  (date -u; soup data lint data/$f.jsonl --format dpo -o logs/preflight_py312/lint_$f.json) 2>&1 | tee logs/preflight_py312/lint_$f.txt; \
done

Wed Sep 30 11:11:23 AM UTC 2026
schema file: /content/venv312/lib/python3.12/site-packages/soup_cli/config/schema.py
root model: SoupConfig
ok       base
ok       task
ok       backend
ok       data
ok       data.train
ok       data.format
ok       data.max_length
ok       data.val_split
ok       training
ok       training.epochs
ok       training.lr
ok       training.batch_size
ok       training.gradient_accumulation_steps
ok       training.quantization
ok       training.gradient_checkpointing
ok       training.stream_layers
ok       training.stream_source
ok       training.stream_buffers
ok       training.dpo_beta
ok       training.seed
ok       training.lora
ok       training.lora.r
ok       training.lora.alpha
ok       training.lora.dropout
ok       training.lora.target_modules
ok       output

--- data (DataConfig) relevant fields: name | default ---
  val_split | 0.1
  raft_shuffle_seed | None
  replay_seed | None
  streaming | False
  eval_on_each_dataset | False

--- training (

In [ ]:
%%bash
(date -u
python - <<'PY'
from soup_cli.utils.gpu import bf16_fp16_flags, resolve_frozen_base_load_dtype

print("bf16/fp16 flags:", bf16_fp16_flags("cuda"))
print("frozen base load dtype:", resolve_frozen_base_load_dtype("cuda"))
PY
) 2>&1 | tee logs/env_py312/precision_resolution.txt

Wed Sep 30 11:15:38 AM UTC 2026
bf16/fp16 flags: (False, True)
frozen base load dtype: torch.float16


In [ ]:
%%bash

SOUP=$(python -c "import soup_cli, os; print(os.path.dirname(soup_cli.__file__))")

(
date -u
echo "SOUP SOURCE: $SOUP"

echo
echo "=== DPO TRAINER: lengths / precision / reference / streaming ==="
grep -n -iE \
"max_prompt_length|max_length|enforce_preference_sequence_limit|bf16_fp16_flags|resolve_frozen_base_load_dtype|materialize_meta_adapter_copy|stream_layers|DPOConfig|beta=" \
"$SOUP/trainer/dpo.py"

echo
echo "=== GPU PRECISION HELPERS ==="
grep -n -A35 -E \
"def bf16_fp16_flags|def resolve_frozen_base_load_dtype" \
"$SOUP/utils/gpu.py"

) 2>&1 | tee logs/preflight_py312/dpo_source_audit.txt

Wed Sep 30 11:16:46 AM UTC 2026
SOUP SOURCE: /content/venv312/lib/python3.12/site-packages/soup_cli

=== DPO TRAINER: lengths / precision / reference / streaming ===
12:    bf16_fp16_flags,
16:    resolve_frozen_base_load_dtype,
76:        from trl import DPOConfig, DPOTrainer
80:            enforce_preference_sequence_limit,
99:        use_streaming = bool(getattr(tcfg, "stream_layers", False))
132:                seq_length=cfg.data.max_length,
166:        _bf16, _fp16 = bf16_fp16_flags(self.device, allow_mps_bf16=True)
167:        dpo_config = DPOConfig(
184:                tcfg.gradient_checkpointing, stream_layers=tcfg.stream_layers
200:            beta=tcfg.dpo_beta,
201:            max_length=cfg.data.max_length,
202:            # #326 — trl dropped `max_prompt_length` from DPOConfig at 0.29.0
203:            # with no successor field; `max_length` absorbs it. Asked of the
206:            **prompt_length_kwargs(DPOConfig, cfg.data.max_length // 2),
219:        if not config_acce

In [ ]:
%%writefile token_audit.py
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_PROMPT = 512
MAX_TOTAL = 1024

tok = AutoTokenizer.from_pretrained(MODEL)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]

def audit(rows, name):
    n = len(rows)

    prompt_over = 0
    chosen_total_over = 0
    rejected_total_over = 0
    either_total_over = 0
    asymmetric_total_over = 0

    prompt_lens = []
    chosen_total_lens = []
    rejected_total_lens = []

    for r in rows:
        p = ids(r["prompt"])
        c = ids(r["chosen"])
        rej = ids(r["rejected"])

        # DPO completions receive EOS if not already present
        if tok.eos_token_id is not None:
            if not c or c[-1] != tok.eos_token_id:
                c = c + [tok.eos_token_id]
            if not rej or rej[-1] != tok.eos_token_id:
                rej = rej + [tok.eos_token_id]

        lp = len(p)
        lc = lp + len(c)
        lr = lp + len(rej)

        prompt_lens.append(lp)
        chosen_total_lens.append(lc)
        rejected_total_lens.append(lr)

        p_over = lp > MAX_PROMPT
        c_over = lc > MAX_TOTAL
        r_over = lr > MAX_TOTAL

        prompt_over += p_over
        chosen_total_over += c_over
        rejected_total_over += r_over
        either_total_over += (c_over or r_over)
        asymmetric_total_over += (c_over != r_over)

    def pct(x):
        return round(100 * x / n, 2)

    def q(xs, p):
        xs = sorted(xs)
        idx = min(len(xs)-1, round((len(xs)-1)*p))
        return xs[idx]

    print(f"\n=== {name} ===")
    print("rows:", n)

    print(
        "prompt tokens p50/p90/p99/max:",
        q(prompt_lens, .50),
        q(prompt_lens, .90),
        q(prompt_lens, .99),
        max(prompt_lens),
    )

    print(
        "chosen total p50/p90/p99/max:",
        q(chosen_total_lens, .50),
        q(chosen_total_lens, .90),
        q(chosen_total_lens, .99),
        max(chosen_total_lens),
    )

    print(
        "rejected total p50/p90/p99/max:",
        q(rejected_total_lens, .50),
        q(rejected_total_lens, .90),
        q(rejected_total_lens, .99),
        max(rejected_total_lens),
    )

    print(f"prompt > {MAX_PROMPT}: {prompt_over}/{n} ({pct(prompt_over)}%)")
    print(f"chosen total > {MAX_TOTAL}: {chosen_total_over}/{n} ({pct(chosen_total_over)}%)")
    print(f"rejected total > {MAX_TOTAL}: {rejected_total_over}/{n} ({pct(rejected_total_over)}%)")
    print(f"either branch > {MAX_TOTAL}: {either_total_over}/{n} ({pct(either_total_over)}%)")
    print(f"asymmetric overflow: {asymmetric_total_over}/{n} ({pct(asymmetric_total_over)}%)")

for split in ["train", "eval"]:
    audit(load_jsonl(f"data/{split}.jsonl"), split)

Writing token_audit.py


In [ ]:
!(date -u; python token_audit.py) 2>&1 | tee logs/preflight_py312/token_audit.txt

Wed Sep 30 11:19:20 AM UTC 2026

=== train ===
rows: 500
prompt tokens p50/p90/p99/max: 161 476 781 1000
chosen total p50/p90/p99/max: 287 657 925 1129
rejected total p50/p90/p99/max: 282 650 951 1099
prompt > 512: 41/500 (8.2%)
chosen total > 1024: 2/500 (0.4%)
rejected total > 1024: 2/500 (0.4%)
either branch > 1024: 2/500 (0.4%)
asymmetric overflow: 0/500 (0.0%)

=== eval ===
rows: 50
prompt tokens p50/p90/p99/max: 159 439 626 626
chosen total p50/p90/p99/max: 278 644 872 872
rejected total p50/p90/p99/max: 287 587 871 871
prompt > 512: 2/50 (4.0%)
chosen total > 1024: 0/50 (0.0%)
rejected total > 1024: 0/50 (0.0%)
either branch > 1024: 0/50 (0.0%)
asymmetric overflow: 0/50 (0.0%)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from transformers import AutoConfig

# ------------------------------------------------------------
# PRE-TRAINING MEMORY BUDGET
# Qwen2.5-1.5B-Instruct + DPO + LoRA r=16 + layer streaming
# This is an estimate, NOT a measurement.
# ------------------------------------------------------------

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Our soup.yaml
BATCH_SIZE = 2
MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 512
LORA_R = 16
STREAM_BUFFERS = 2

# DPO concatenates chosen + rejected
EFFECTIVE_SEQUENCES = BATCH_SIZE * 2

cfg = AutoConfig.from_pretrained(MODEL)

H = cfg.hidden_size
I = cfg.intermediate_size
N_LAYERS = cfg.num_hidden_layers
N_HEADS = cfg.num_attention_heads
N_KV_HEADS = cfg.num_key_value_heads
VOCAB = cfg.vocab_size
HEAD_DIM = getattr(cfg, "head_dim", H // N_HEADS)
KV_OUT = N_KV_HEADS * HEAD_DIM
Q_OUT = N_HEADS * HEAD_DIM

ATTN_BIAS = bool(getattr(cfg, "attention_bias", True))
TIED_EMBEDDINGS = bool(getattr(cfg, "tie_word_embeddings", True))

MiB = 1024**2
GiB = 1024**3

def mib(x):
    return x / MiB

def gib(x):
    return x / GiB


# ------------------------------------------------------------
# 1. BASE MODEL PARAMETER COUNT
# ------------------------------------------------------------

# Qwen2 attention
q_params = H * Q_OUT + (Q_OUT if ATTN_BIAS else 0)
k_params = H * KV_OUT + (KV_OUT if ATTN_BIAS else 0)
v_params = H * KV_OUT + (KV_OUT if ATTN_BIAS else 0)
o_params = Q_OUT * H

# SwiGLU MLP: gate_proj, up_proj, down_proj
mlp_params = (H * I) + (H * I) + (I * H)

# two RMSNorms per decoder layer
norm_params = 2 * H

params_per_layer = (
    q_params + k_params + v_params + o_params
    + mlp_params + norm_params
)

embedding_params = VOCAB * H
final_norm_params = H

# Qwen ties lm_head to embedding by default
lm_head_extra_params = 0 if TIED_EMBEDDINGS else VOCAB * H

base_params = (
    embedding_params
    + N_LAYERS * params_per_layer
    + final_norm_params
    + lm_head_extra_params
)

full_base_fp16 = base_params * 2


# ------------------------------------------------------------
# 2. LAYER STREAMING
# ------------------------------------------------------------

layer_fp16 = params_per_layer * 2

# Working assumption:
# non-layer embeddings/final norm stay resident on GPU,
# while STREAM_BUFFERS decoder layers are resident at a time.
streamed_base_gpu = (
    embedding_params * 2
    + final_norm_params * 2
    + lm_head_extra_params * 2
    + STREAM_BUFFERS * layer_fp16
)


# ------------------------------------------------------------
# 3. LoRA PARAMETERS
# Targets:
# q, k, v, o, gate, up, down
# LoRA params for a Linear = r * (in_features + out_features)
# ------------------------------------------------------------

lora_per_layer = LORA_R * (
    (H + Q_OUT)       # q
    + (H + KV_OUT)    # k
    + (H + KV_OUT)    # v
    + (Q_OUT + H)     # o
    + (H + I)         # gate
    + (H + I)         # up
    + (I + H)         # down
)

lora_params = lora_per_layer * N_LAYERS

# Conservative assumption: trainable LoRA tensors are fp32
lora_weights = lora_params * 4
lora_gradients = lora_params * 4

# AdamW m + v states, fp32
optimizer_states = lora_params * 4 * 2

# DPO reference adapter snapshot.
# Soup streaming does NOT need a second full base model.
reference_adapter = lora_params * 4


# ------------------------------------------------------------
# 4. ACTIVATIONS
# ------------------------------------------------------------

# One fp16 hidden-state boundary for effective DPO batch
one_hidden = (
    EFFECTIVE_SEQUENCES
    * MAX_LENGTH
    * H
    * 2
)

# Approximate checkpoint boundary storage if one input activation
# is retained per decoder layer.
checkpoint_boundaries = one_hidden * N_LAYERS

# Approximate transient attention score tensor if materialized:
# [B_eff, heads, seq, seq], fp16.
# SDPA/Flash-style kernels can make actual memory smaller.
attention_scores = (
    EFFECTIVE_SEQUENCES
    * N_HEADS
    * MAX_LENGTH
    * MAX_LENGTH
    * 2
)

# gate + up projections in the MLP, fp16
mlp_gate_up = (
    EFFECTIVE_SEQUENCES
    * MAX_LENGTH
    * I
    * 2                 # gate + up
    * 2                 # fp16 bytes
)


# ------------------------------------------------------------
# 5. LOGITS
# ------------------------------------------------------------

# Conservative upper bound:
# [chosen+rejected batch, 1024 positions, vocab], fp16
#
# TRL/Qwen may retain only completion logits, making actual usage lower,
# but we keep the full capped length as a safe pre-run upper bound.
logits_upper = (
    EFFECTIVE_SEQUENCES
    * MAX_LENGTH
    * VOCAB
    * 2
)


# ------------------------------------------------------------
# 6. WORKING PEAK ESTIMATE
# ------------------------------------------------------------

persistent = (
    streamed_base_gpu
    + lora_weights
    + lora_gradients
    + optimizer_states
    + reference_adapter
)

# Conservative: count both major transient activation terms,
# checkpoint boundaries and the logits upper bound.
accounted_peak = (
    persistent
    + checkpoint_boundaries
    + attention_scores
    + mlp_gate_up
    + logits_upper
)

# CUDA context, allocator fragmentation, kernels, temporary optimizer
# buffers and framework allocations are not captured exactly.
overhead_low = 1.0 * GiB
overhead_high = 2.5 * GiB

predicted_low = accounted_peak + overhead_low
predicted_high = accounted_peak + overhead_high


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

report = f"""
# Pre-training memory budget

Timestamp: {timestamp}

**This estimate was recorded before `soup profile`, dry-run, or training.**

## Configuration

- Model: {MODEL}
- Hidden size: {H}
- Intermediate size: {I}
- Layers: {N_LAYERS}
- Attention heads / KV heads: {N_HEADS} / {N_KV_HEADS}
- Vocabulary: {VOCAB}
- Batch size: {BATCH_SIZE}
- DPO effective chosen+rejected sequences: {EFFECTIVE_SEQUENCES}
- max_length: {MAX_LENGTH}
- max_prompt_length: {MAX_PROMPT_LENGTH}
- LoRA rank: {LORA_R}
- stream_buffers: {STREAM_BUFFERS}
- quantization: none
- compute/base dtype on T4: fp16

## 1. Base weights

Calculated base parameters: {base_params:,} ({base_params/1e9:.3f} B)

Full resident fp16 base:
{base_params:,} params × 2 bytes
= {gib(full_base_fp16):.3f} GiB

One decoder layer:
{params_per_layer:,} params × 2 bytes
= {mib(layer_fp16):.1f} MiB

Two streamed layer buffers:
2 × {mib(layer_fp16):.1f}
= {mib(STREAM_BUFFERS * layer_fp16):.1f} MiB

Embeddings / non-layer resident working estimate:
embedding = {mib(embedding_params * 2):.1f} MiB

Estimated streamed base resident on GPU:
≈ {mib(streamed_base_gpu):.1f} MiB
= {gib(streamed_base_gpu):.3f} GiB

The full fp16 base still needs roughly {gib(full_base_fp16):.3f} GiB of host-side
storage for RAM streaming, excluding framework/cache overhead.

## 2. LoRA + optimizer state

LoRA trainable parameters:
{lora_params:,}

LoRA weights, fp32:
{lora_params:,} × 4 bytes
= {mib(lora_weights):.1f} MiB

LoRA gradients, fp32:
= {mib(lora_gradients):.1f} MiB

AdamW m + v states:
{lora_params:,} × 8 bytes
= {mib(optimizer_states):.1f} MiB

DPO frozen reference adapter snapshot:
≈ {mib(reference_adapter):.1f} MiB

Adapter/gradient/optimizer/reference subtotal:
≈ {mib(lora_weights + lora_gradients + optimizer_states + reference_adapter):.1f} MiB

## 3. Activations

DPO processes chosen and rejected together, so batch 2 becomes
approximately 4 sequences for the concatenated forward.

One fp16 hidden-state tensor at max length:
4 × 1024 × {H} × 2 bytes
= {mib(one_hidden):.1f} MiB

Approx. checkpoint boundaries over {N_LAYERS} layers:
{N_LAYERS} × {mib(one_hidden):.1f}
= {mib(checkpoint_boundaries):.1f} MiB

Potential attention-score transient:
≈ {mib(attention_scores):.1f} MiB
(actual SDPA implementation may use less)

MLP gate+up transient:
≈ {mib(mlp_gate_up):.1f} MiB

## 4. Logits

Conservative maximum fp16 logits tensor:

4 × 1024 × {VOCAB:,} × 2 bytes
= {gib(logits_upper):.3f} GiB

TRL/Qwen can avoid retaining logits for unnecessary prompt positions,
so actual logits memory may be lower. This is intentionally an upper bound.

## 5. DPO reference pass

Soup's streamed DPO path uses the same frozen base with a reference
adapter snapshot rather than keeping a second full {base_params/1e9:.3f}B
base model in VRAM.

The reference forward is no-grad and occurs separately, so it increases
runtime and transient forward memory but should not double policy
activations/optimizer state at peak.

## 6. Predicted peak VRAM

Explicitly accounted tensors:
≈ {gib(accounted_peak):.2f} GiB

Allowing ~1.0–2.5 GiB for CUDA context, allocator fragmentation,
temporary kernels/optimizer buffers, and unmodelled framework tensors:

**Predicted peak VRAM: approximately {gib(predicted_low):.1f}–{gib(predicted_high):.1f} GiB**

T4 capacity observed by Soup doctor: ~14.6 GiB.

A temporary full-model load during streaming setup could increase the peak;
that is one reason the measured peak may exceed this steady-state estimate.

## What this estimate does NOT prove

This is a pre-run analytical budget, not evidence that Soup actually
streams correctly or that the model trains correctly. It will be compared
against `soup profile`, raw `nvidia-smi`, and measured CUDA peak memory.
"""

Path("logs").mkdir(exist_ok=True)
Path("logs/memory_budget_before.md").write_text(report, encoding="utf-8")

print(report)
print("\nSaved to: logs/memory_budget_before.md")


# Pre-training memory budget

Timestamp: 2026-09-30 11:29:32 UTC

**This estimate was recorded before `soup profile`, dry-run, or training.**

## Configuration

- Model: Qwen/Qwen2.5-1.5B-Instruct
- Hidden size: 1536
- Intermediate size: 8960
- Layers: 28
- Attention heads / KV heads: 12 / 2
- Vocabulary: 151936
- Batch size: 2
- DPO effective chosen+rejected sequences: 4
- max_length: 1024
- max_prompt_length: 512
- LoRA rank: 16
- stream_buffers: 2
- quantization: none
- compute/base dtype on T4: fp16

## 1. Base weights

Calculated base parameters: 1,543,714,304 (1.544 B)

Full resident fp16 base:
1,543,714,304 params × 2 bytes
= 2.875 GiB

One decoder layer:
46,797,824 params × 2 bytes
= 89.3 MiB

Two streamed layer buffers:
2 × 89.3
= 178.5 MiB

Embeddings / non-layer resident working estimate:
embedding = 445.1 MiB

Estimated streamed base resident on GPU:
≈ 623.6 MiB
= 0.609 GiB

The full fp16 base still needs roughly 2.875 GiB of host-side
storage for RAM streaming, excluding 

In [ ]:
!(date -u; soup profile -c soup.yaml) 2>&1 | tee logs/preflight_py312/soup_profile.txt

Wed Sep 30 11:30:41 AM UTC 2026
╭────────────────────────────── Training Profile ──────────────────────────────╮
│ Model:     Qwen/Qwen2.5-1.5B-Instruct                                        │
│ Params:    1.5B (trainable: 12,845,056 with LoRA r=16)                       │
│ Quantization: none                                                           │
│ Gradient checkpointing: enabled                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

GPU Memory Estimate:
  Model                              ~3.0 GB  
  LoRA                               ~0.0 GB  
  Optimizer                          ~0.1 GB  
  Activations (bs=2, seq=1024)       ~0.2 GB  
  Overhead                           ~1.5 GB  
  --------------------            ----------  
  Total                              ~4.7 GB  

╭─────────────────────────────── Speed Estimate ───────────────────────────────╮
│ Tokens/sec: ~1,250                             

In [ ]:
%%bash

SOUP=$(python -c "import soup_cli, os; print(os.path.dirname(soup_cli.__file__))")

echo "=== PROFILE / LoRA PARAM ESTIMATION ==="

grep -RniE \
"trainable.*lora|lora.*trainable|lora_params|trainable_params|profile" \
"$SOUP" | head -80

=== PROFILE / LoRA PARAM ESTIMATION ===
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:322:class ThroughputProfile(_FrozenArtifact):
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:325:    profile_sha256: str
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:338:        "profile_sha256",
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:343:    def _profile_digests(cls, value: str, info) -> str:
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:357:    def _ordered_range(self) -> ThroughputProfile:
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:366:    status: Literal["unknown", "profiled"]
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:369:    profile_sha256: str | None
/content/venv312/lib/python3.12/site-packages/soup_cli/autodistill/contract.py:376:                for val

In [ ]:
!(date -u; \
python -c "import trl,inspect; p=inspect.signature(trl.DPOConfig).parameters; print('trl',trl.__version__); print({k:(p[k].default if k in p else 'ABSENT') for k in ['max_prompt_length','max_completion_length','max_length','truncation_mode','padding_free','precompute_ref_log_probs']})"; \
soup env check; \
python -m pip check) 2>&1 | tee logs/preflight_py312/trl_prompt_cap_and_deps.txt

Wed Sep 30 11:40:38 AM UTC 2026
trl 0.29.1
{'max_prompt_length': 'ABSENT', 'max_completion_length': 'ABSENT', 'max_length': 1024, 'truncation_mode': 'keep_start', 'padding_free': False, 'precompute_ref_log_probs': False}
No lock file at soup-env.lock; run `soup env lock` first.
/content/venv312/bin/python: No module named pip


In [ ]:
%%bash
SOUP=$(python -c "import soup_cli,os; print(os.path.dirname(soup_cli.__file__))")

(
date -u
grep -n -A45 -B10 \
"def enforce_preference_sequence_limit" \
"$SOUP/trainer/_trl_compat.py"

echo
grep -n -A25 -B8 \
"if not config_accepts(DPOConfig, \"max_prompt_length\")" \
"$SOUP/trainer/dpo.py"
) 2>&1 | tee logs/preflight_py312/prompt_cap_source.txt

Wed Sep 30 11:44:30 AM UTC 2026
124-    """Truncate one token sequence using TRL's preference-side convention."""
125-    if limit <= 0:
126-        return []
127-    if len(tokens) <= limit:
128-        return tokens
129-    if mode == "keep_end":
130-        return tokens[-limit:]
131-    return tokens[:limit]
132-
133-
134:def enforce_preference_sequence_limit(
135-    dataset: Any,
136-    *,
137-    max_length: int,
138-    max_prompt_length: int,
139-    truncation_mode: str,
140-) -> Any:
141-    """Restore TRL's removed prompt cap on an already-tokenized dataset.
142-
143-    TRL 0.29 exposes two preference dataset layouts. DPO stores a shared
144-    ``prompt_ids`` plus separate completion ids; experimental ORPO stores two
145-    combined sequences whose prompt span is identified by ``-100`` labels.
146-    Applying the cap after TRL tokenizes keeps text and conversational inputs
147-    on the exact same chat-template path while guaranteeing that the tensors
148-    reaching

In [ ]:
!SOUP=$(python -c "import soup_cli,os;print(os.path.dirname(soup_cli.__file__))"); \
 (date -u; echo "== truncation key in schema?"; grep -n -iE "truncation" $SOUP/config/schema.py; \
  echo "== is truncation_mode passed to DPOConfig from our config?"; grep -n -iE "truncation_mode|DPOConfig\(" $SOUP/trainer/dpo.py; \
  echo "== how DPO rows are rendered (text vs messages/chat template)"; grep -rn -iE "chosen|conversational|messages|apply_chat_template" $SOUP/data/ | grep -iE "dpo|prefer|chosen" | head -30) \
  2>&1 | tee logs/preflight_py312/truncation_and_rendering.txt

Wed Sep 30 11:48:31 AM UTC 2026
== truncation key in schema?
3450:                "(the truncation mask is derived from the importance-sampling "
== is truncation_mode passed to DPOConfig from our config?
167:        dpo_config = DPOConfig(
226:                "truncation_mode": dpo_config.truncation_mode,
== how DPO rows are rendered (text vs messages/chat template)
grep: /content/venv312/lib/python3.12/site-packages/soup_cli/data/__pycache__/loss_mask.cpython-312.pyc: binary file matches
grep: /content/venv312/lib/python3.12/site-packages/soup_cli/data/__pycache__/loader.cpython-312.pyc: binary file matches
grep: /content/venv312/lib/python3.12/site-packages/soup_cli/data/__pycache__/formats.cpython-312.pyc: binary file matches
/content/venv312/lib/python3.12/site-packages/soup_cli/data/loader.py:648:    streaming=True)``, where ``builder`` is chosen per entry from its file
/content/venv312/lib/python3.12/site-packages/soup_cli/data/formats.py:7:- dpo: {"prompt": ..., "chosen": ..., 

In [ ]:
!(date -u; \
soup env lock; \
soup env check; \
echo "exit=$?"; \
uv pip check --python /content/venv312/bin/python) \
2>&1 | tee logs/preflight_py312/deps_check.txt

Wed Sep 30 11:48:40 AM UTC 2026
╭────────────────────────────────── env lock ──────────────────────────────────╮
│ Locked 9 packages to soup-env.lock                                           │
│ Python: 3.12.3 | Platform: linux-x86_64 | CUDA: 12.8.1                       │
╰──────────────────────────────────────────────────────────────────────────────╯
╭───────────────────────────────── env check ──────────────────────────────────╮
│ ABI-clean. No drift detected.                                                │
╰──────────────────────────────────────────────────────────────────────────────╯
exit=0
Using Python 3.12.3 environment at: venv312
Checked 89 packages in 2ms
All installed packages are compatible


In [ ]:
%%bash

mkdir -p logs/preflight_py312

date -u

python - <<'PY' 2>&1 | tee logs/preflight_py312/trl_rendering_check.txt
import json
import trl

from trl.data_utils import is_conversational, maybe_apply_chat_template
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

with open("data/train.jsonl", encoding="utf-8") as f:
    row = json.loads(f.readline())

print("trl", trl.__version__, "| is_conversational:", is_conversational(row))

out = maybe_apply_chat_template(row, tok)

print("prompt changed by template:", out["prompt"] != row["prompt"])
print("prompt tail:", repr(out["prompt"][-80:]))
print("chosen head:", repr(out["chosen"][:60]))
print("eos:", repr(tok.eos_token))
PY

Wed Sep 30 11:55:22 AM UTC 2026
trl 0.29.1 | is_conversational: False
prompt changed by template: False
prompt tail: 'Человек: Можешь помочь мне написать страшную историю?\n\nАссистент:'
chosen head: 'Конечно, буду рад помочь! Давайте обсудим некоторые идеи для'
eos: '<|im_end|>'


In [ ]:
!mkdir -p data/superseded_v1 && \
cp -v data/train.jsonl data/eval.jsonl data/manifest.json data/superseded_v1/ && \
sha256sum data/superseded_v1/* | tee data/superseded_v1/SHA256SUMS.txt

'data/train.jsonl' -> 'data/superseded_v1/train.jsonl'
'data/eval.jsonl' -> 'data/superseded_v1/eval.jsonl'
'data/manifest.json' -> 'data/superseded_v1/manifest.json'
9030d5b48edf6e56519ab95758730d88f2e2be17c63161a5dac44d086faac753  data/superseded_v1/eval.jsonl
91506b4e2beff0c4d7171deb93594ffe7084c9812ebf8456b20062ae384fe95a  data/superseded_v1/manifest.json
ee850055e6724e1f50d06aeac628ff16a55a3b0ef27c9ee610cb232dc9b106ac  data/superseded_v1/train.jsonl


In [ ]:
from huggingface_hub import HfApi
print(HfApi().model_info("Qwen/Qwen2.5-1.5B-Instruct").sha)

989aa7980e4cf806f80c7fef2b1adb7bc71aa306


In [ ]:
%%writefile prepare_data.py

import json
import re
import hashlib
from collections import Counter
from pathlib import Path

import numpy as np
import transformers
import tokenizers
from datasets import load_dataset
from transformers import AutoTokenizer


# ============================================================
# Reproducibility
# ============================================================

REPO = "d0rj/full-hh-rlhf-ru"
DATA_REV = "fd4a069df1b99329527389101f48540140478ff7"
SOURCE_SPLIT = "train"

SEED = 42

N_EVAL = 50
N_TRAIN = 500

MAX_CHARS = 3000
MIN_CYR = 0.5


# ============================================================
# Tokenizer pinning for v2 token-length filters
# ============================================================

TOK_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TOK_REV = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

TOK = AutoTokenizer.from_pretrained(
    TOK_ID,
    revision=TOK_REV,
)

# Raw-text rendering, matching the experimentally verified TRL 0.29.1 path:
# is_conversational=False, chat template not applied.
#
# Soup limits:
#   prompt <= 512
#   total  <= 1024
#
# We keep a small safety margin so final rows should not need Soup truncation.
PROMPT_TOK_MAX = 500
TOTAL_TOK_MAX = 1000
EOS_BUDGET = 1


def ntok(text):
    return len(
        TOK(
            text,
            add_special_tokens=False,
        )["input_ids"]
    )


# ============================================================
# Helpers
# ============================================================
def trl_boundary_mismatch(prompt, completion):
    """Match TRL 0.29.1 raw-text DPO preprocessing at the prompt/completion boundary."""
    assert tok.eos_token is not None

    completion_with_eos = (
        completion
        if completion.endswith(tok.eos_token)
        else completion + tok.eos_token
    )

    prompt_ids = tok(prompt)["input_ids"]
    full_ids = tok(prompt + completion_with_eos)["input_ids"]

    return full_ids[:len(prompt_ids)] != prompt_ids

def norm(text):
    """Whitespace-normalized string for exact overlap checks."""
    return re.sub(r"\s+", " ", text).strip()


def cyr_ratio(text):
    """
    Fraction of alphabetic characters that are Cyrillic.
    Non-letter punctuation/whitespace is ignored.
    """
    letters = re.findall(r"[A-Za-zА-Яа-яЁё]", text)

    if not letters:
        return 0.0

    cyr = sum(
        bool(re.fullmatch(r"[А-Яа-яЁё]", ch))
        for ch in letters
    )

    return cyr / len(letters)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


def char_stats(rows):
    result = {}

    for field in ("prompt", "chosen", "rejected"):
        values = [len(row[field]) for row in rows]

        result[field] = {
            "p50": int(np.percentile(values, 50)),
            "p90": int(np.percentile(values, 90)),
            "p99": int(np.percentile(values, 99)),
            "max": int(max(values)),
        }

    result["frac_chosen_longer"] = float(
        np.mean(
            [
                len(row["chosen"]) > len(row["rejected"])
                for row in rows
            ]
        )
    )

    return result


def tokq(pairs):
    def q(values):
        return {
            str(p): int(np.percentile(values, p))
            for p in (50, 90, 99, 100)
        }

    return {
        "prompt_tok": q([p for p, _ in pairs]),
        "total_tok_incl_eos": q([t for _, t in pairs]),
    }


def last_nonempty_line(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    return lines[-1] if lines else ""


# ============================================================
# Load exact pinned source revision
# ============================================================

print("Loading dataset...")

ds = load_dataset(
    REPO,
    revision=DATA_REV,
    split=SOURCE_SPLIT,
)

print("source rows:", len(ds))

# Deterministic order.
ds = ds.shuffle(seed=SEED)


# ============================================================
# Build held-out eval first, then train
# ============================================================

eval_rows = []
train_rows = []

seen_prompts = set()

# All completions from held-out eval.
# Train rows are excluded if either completion exactly overlaps one of these.
eval_completions = set()

drops = Counter()

tokstats = {
    "eval": [],
    "train": [],
}

rows_scanned = 0


for source_row in ds:
    rows_scanned += 1

    p = source_row.get("prompt", "")
    c = source_row.get("chosen", "")
    r = source_row.get("rejected", "")

    # --------------------------------------------------------
    # R1 - empty chosen/rejected
    # --------------------------------------------------------
    if not c.strip() or not r.strip():
        drops["R1_empty"] += 1
        continue

    # --------------------------------------------------------
    # R2 - identical chosen/rejected
    # --------------------------------------------------------
    if norm(c) == norm(r):
        drops["R2_identical"] += 1
        continue

    # --------------------------------------------------------
    # R3 - Russian/Cyrillic content requirement
    # --------------------------------------------------------
    if cyr_ratio(c) < MIN_CYR or cyr_ratio(r) < MIN_CYR:
        drops["R3_low_cyrillic"] += 1
        continue

    # --------------------------------------------------------
    # R4 - provisional character-length filter
    # --------------------------------------------------------
    if len(p) + max(len(c), len(r)) > MAX_CHARS:
        drops["R4_over_3000_chars"] += 1
        continue

    prompt_key = norm(p)

    # --------------------------------------------------------
    # R5 - exact normalized prompt duplicate
    # --------------------------------------------------------
    if prompt_key in seen_prompts:
        drops["R5_dup_exact_prompt"] += 1
        continue

    # --------------------------------------------------------
    # R7 / R8 - v2 tokenizer-based length filters
    # --------------------------------------------------------

    pt = ntok(p)

    chosen_tok = ntok(c) + EOS_BUDGET
    rejected_tok = ntok(r) + EOS_BUDGET

    tt = pt + max(
        chosen_tok,
        rejected_tok,
    )

    # R7: stay safely below Soup's 512-token prompt cap.
    if pt > PROMPT_TOK_MAX:
        drops["R7_prompt_over_500_tok"] += 1
        continue

    # R8: stay safely below Soup's 1024-token sequence cap.
    if tt > TOTAL_TOK_MAX:
        drops["R8_total_over_1000_tok"] += 1
        continue

    # R9: tokenization boundary must match TRL preprocessing.
    if trl_boundary_mismatch(p, c) or trl_boundary_mismatch(p, r):
      drops["R9_boundary_tokenization_mismatch"] += 1
      continue

    # --------------------------------------------------------
    # R6 - exact normalized completion overlap with eval
    #
    # Only applies once held-out eval is complete.
    # --------------------------------------------------------
    if len(eval_rows) >= N_EVAL:
        if (
            norm(c) in eval_completions
            or norm(r) in eval_completions
        ):
            drops["R6_completion_overlap_with_eval"] += 1
            continue

    row = {
        "prompt": p,
        "chosen": c,
        "rejected": r,
    }

    # --------------------------------------------------------
    # First 50 eligible rows -> held-out evaluation
    # --------------------------------------------------------
    if len(eval_rows) < N_EVAL:
        seen_prompts.add(prompt_key)
        eval_rows.append(row)

        eval_completions.update(
            [
                norm(c),
                norm(r),
            ]
        )

        tokstats["eval"].append(
            (
                pt,
                tt,
            )
        )

        continue

    # --------------------------------------------------------
    # Next 500 eligible rows -> train
    # --------------------------------------------------------
    seen_prompts.add(prompt_key)
    train_rows.append(row)

    tokstats["train"].append(
        (
            pt,
            tt,
        )
    )

    if len(train_rows) >= N_TRAIN:
        break


# ============================================================
# Hard assertions
# ============================================================

assert len(eval_rows) == N_EVAL, (
    f"Expected {N_EVAL} eval rows, got {len(eval_rows)}"
)

assert len(train_rows) == N_TRAIN, (
    f"Expected {N_TRAIN} train rows, got {len(train_rows)}"
)

all_tokstats = (
    tokstats["train"]
    + tokstats["eval"]
)

assert max(
    p
    for p, _ in all_tokstats
) <= PROMPT_TOK_MAX

assert max(
    t
    for _, t in all_tokstats
) <= TOTAL_TOK_MAX


# ============================================================
# Train/eval exact-overlap audit
# ============================================================

train_prompts = {
    norm(x["prompt"])
    for x in train_rows
}

eval_prompts = {
    norm(x["prompt"])
    for x in eval_rows
}

train_chosen = {
    norm(x["chosen"])
    for x in train_rows
}

eval_chosen = {
    norm(x["chosen"])
    for x in eval_rows
}

train_rejected = {
    norm(x["rejected"])
    for x in train_rows
}

eval_rejected = {
    norm(x["rejected"])
    for x in eval_rows
}

train_all_completions = (
    train_chosen
    | train_rejected
)

eval_all_completions = (
    eval_chosen
    | eval_rejected
)

overlap = {
    "prompt": len(
        train_prompts
        & eval_prompts
    ),
    "chosen": len(
        train_chosen
        & eval_chosen
    ),
    "rejected": len(
        train_rejected
        & eval_rejected
    ),
    "any_completion": len(
        train_all_completions
        & eval_all_completions
    ),
}

assert overlap["prompt"] == 0
assert overlap["chosen"] == 0
assert overlap["rejected"] == 0
assert overlap["any_completion"] == 0


# ============================================================
# Assistant-marker audit
# ============================================================

assistant_end_re = re.compile(
    r"(?:Ассистент|Помощник|Assistant)\s*:\s*$"
)

prompts_ending_with_assistant_marker = {
    "train": sum(
        bool(
            assistant_end_re.search(
                x["prompt"]
            )
        )
        for x in train_rows
    ),
    "eval": sum(
        bool(
            assistant_end_re.search(
                x["prompt"]
            )
        )
        for x in eval_rows
    ),
}

top_prompt_endings = {
    "train": Counter(
        last_nonempty_line(
            x["prompt"]
        )
        for x in train_rows
    ).most_common(10),

    "eval": Counter(
        last_nonempty_line(
            x["prompt"]
        )
        for x in eval_rows
    ).most_common(10),
}


# ============================================================
# Role marker counts
# ============================================================

role_markers = [
    "Человек",
    "Ассистент",
    "Помощник",
    "Human",
    "Assistant",
]

all_selected_text = "\n".join(
    row[field]
    for row in (
        eval_rows
        + train_rows
    )
    for field in (
        "prompt",
        "chosen",
        "rejected",
    )
)

role_marker_counts = {
    marker: len(
        re.findall(
            rf"{re.escape(marker)}\s*:",
            all_selected_text,
        )
    )
    for marker in role_markers
}


# ============================================================
# Write final v2 files
# ============================================================

OUT = Path("data")
OUT.mkdir(
    parents=True,
    exist_ok=True,
)

train_path = OUT / "train.jsonl"
eval_path = OUT / "eval.jsonl"
manifest_path = OUT / "manifest.json"

write_jsonl(
    train_path,
    train_rows,
)

write_jsonl(
    eval_path,
    eval_rows,
)

train_sha = sha256_file(train_path)
eval_sha = sha256_file(eval_path)


# ============================================================
# Manifest
# ============================================================

# Make sure zero-count rules are also explicitly visible.
drop_keys = [
    "R1_empty",
    "R2_identical",
    "R3_low_cyrillic",
    "R4_over_3000_chars",
    "R5_dup_exact_prompt",
    "R6_completion_overlap_with_eval",
    "R7_prompt_over_500_tok",
    "R8_total_over_1000_tok",
    "R9_boundary_tokenization_mismatch",
]

drop_manifest = {
    key: int(drops[key])
    for key in drop_keys
}

manifest = {
    "repo": REPO,
    "revision": DATA_REV,
    "source_split": SOURCE_SPLIT,
    "seed": SEED,

    "rows_scanned": rows_scanned,

    "drops": drop_manifest,

    "drop_counter_scope": (
        "Drop counts apply only to rows scanned until "
        "50 eval + 500 train rows were collected; "
        "they are not full-dataset prevalence estimates."
    ),

    "selection": {
        "eval": (
            "first 50 eligible rows after deterministic shuffle"
        ),
        "train": (
            "next 500 eligible rows after deterministic shuffle; "
            "train completions are excluded if they exactly overlap eval completions"
        ),
    },

    "filters": {
        "MAX_CHARS": MAX_CHARS,
        "MIN_CYR": MIN_CYR,
    },

    "tokenizer": {
        "id": TOK_ID,
        "revision": TOK_REV,
        "transformers": transformers.__version__,
        "tokenizers": tokenizers.__version__,

        "rendering": (
            "raw text; experimentally verified with TRL 0.29.1: "
            "is_conversational=False and chat template not applied"
        ),

        "rules": {
            "PROMPT_TOK_MAX": PROMPT_TOK_MAX,
            "TOTAL_TOK_MAX": TOTAL_TOK_MAX,
            "EOS_BUDGET": EOS_BUDGET,
            "add_special_tokens": False,
        },
    },

    "eval": {
        "n": len(eval_rows),
        "sha256": eval_sha,
        "char_stats": char_stats(eval_rows),
    },

    "train": {
        "n": len(train_rows),
        "sha256": train_sha,
        "char_stats": char_stats(train_rows),
    },

    "token_stats": {
        "train": tokq(
            tokstats["train"]
        ),
        "eval": tokq(
            tokstats["eval"]
        ),
    },

    "train_eval_overlap": overlap,

    "prompts_ending_with_assistant_marker": (
        prompts_ending_with_assistant_marker
    ),

    "top_prompt_endings": top_prompt_endings,

    "role_marker_counts_combined": role_marker_counts,

    "superseded_note": (
        "v1 sample without token rules is preserved in "
        "data/superseded_v1/. "
        "Initial audit found 41/500 train prompts above Soup's "
        "512-token prompt cap; Soup 0.75.1 fallback uses keep_start, "
        "so those prompts were at risk of tail truncation. "
        "v2 adds conservative 500/1000 token filters before training."
    ),
}

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# Human-readable output
# ============================================================

print(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("WROTE:")
print(train_path)
print(eval_path)
print(manifest_path)

Overwriting prepare_data.py


In [ ]:
!python -m py_compile prepare_data.py && echo "SYNTAX OK"

SYNTAX OK


In [ ]:
!python -m py_compile prepare_data.py && echo "SYNTAX OK"

SYNTAX OK


In [ ]:
!mkdir -p logs/data_v2
!(date -u; which python; python prepare_data.py) 2>&1 | tee logs/data_v2/prepare_data_v2.log

Wed Sep 30 12:32:25 PM UTC 2026
/content/venv312/bin/python
Loading dataset...
source rows: 112052
{
  "repo": "d0rj/full-hh-rlhf-ru",
  "revision": "fd4a069df1b99329527389101f48540140478ff7",
  "source_split": "train",
  "seed": 42,
  "rows_scanned": 970,
  "drops": {
    "R1_empty": 1,
    "R2_identical": 6,
    "R3_low_cyrillic": 319,
    "R4_over_3000_chars": 37,
    "R5_dup_exact_prompt": 0,
    "R6_completion_overlap_with_eval": 4,
    "R7_prompt_over_500_tok": 53,
    "R8_total_over_1000_tok": 0
  },
  "drop_counter_scope": "Drop counts apply only to rows scanned until 50 eval + 500 train rows were collected; they are not full-dataset prevalence estimates.",
  "selection": {
    "eval": "first 50 eligible rows after deterministic shuffle",
    "train": "next 500 eligible rows after deterministic shuffle; train completions are excluded if they exactly overlap eval completions"
  },
  "filters": {
    "MAX_CHARS": 3000,
    "MIN_CYR": 0.5
  },
  "tokenizer": {
    "id": "Qwen/Qwen

In [ ]:
!for f in train eval; do \
  (date -u; soup data validate data/$f.jsonl --format dpo) 2>&1 | tee logs/data_v2/validate_$f.txt; \
  (date -u; soup data lint data/$f.jsonl --format dpo -o logs/data_v2/lint_$f.json) 2>&1 | tee logs/data_v2/lint_$f.txt; \
done

Wed Sep 30 12:35:00 PM UTC 2026
Dataset is valid!

500/500 rows valid for dpo format
Wed Sep 30 12:35:05 PM UTC 2026
                              soup data lint (dpo)                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check           ┃ Verdict ┃ Message                ┃ Evidence                ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ length_bias     │ OK      │ effect size (Cohen's   │ mean chosen=54.5, mean  │
│                 │         │ d) = 0.114 (chosen     │ rejected=49.7           │
│                 │         │ longer than rejected); │                         │
│                 │         │ chosen is longer in    │                         │
│                 │         │ 54.2% of rows          │                         │
│ identical_pairs │ OK      │ 0/500 rows (0.0%) have │                         │
│                 │         │ chosen == rejected     │                   

In [ ]:
!(date -u; soup --log-level verbose train -c soup.yaml --dry-run) \
2>&1 | tee logs/preflight_py312/dry_run.txt

Wed Sep 30 12:36:43 PM UTC 2026
Loading config from soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   Qwen/Qwen2.5-1.5B-Instruct                                          │
│ Task:    dpo                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=16, alpha=32                                                      │
│ Quant:   none                                                                │
│ Seed:    42                                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
Dry run - validating data...
Data OK: 500 train samples
Config valid. Ready to train!


In [ ]:
%%bash

mkdir -p logs/preflight_py312

(date -u; nvidia-smi) 2>&1 | tee logs/preflight_py312/nvidia_smi_before_training.txt

Wed Sep 30 12:44:01 PM UTC 2026
Wed Sep 30 12:44:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

In [ ]:
%%writefile verify_training.py
#!/usr/bin/env python
"""Part 2 verification: held-out DPO preference margins, before vs after training.

ONE scoring path for every mode, reproducing TRL 0.29.1 DPOTrainer preprocessing for
non-conversational rows (verified against TRL source):
  1. raw text, NO chat template (TRL 0.29.1: is_conversational=False, template not applied);
  2. append tok.eos_token to chosen/rejected TEXT if absent;
  3. prompt_ids = tok(prompt); full_ids = tok(prompt + completion_with_eos);
  4. assert full_ids starts with prompt_ids; completion_ids = full_ids[len(prompt_ids):];
  5. logp(completion | prompt) = SUM of token log-probs over completion_ids only (causal shift).
Soup 0.75.1 cap_dpo (prompt 512 / total 1024) is asserted to be a no-op for every row.

Modes:
  --baseline          base model on data/eval.jsonl -> artifacts/baseline_logps.json
                      + negative control 1: zero-initialised LoRA (lora_B=0) -> must be NOT_TRAINED
  --adapter PATH      trained adapter -> artifacts/post_logps.json + verdict vs baseline
                      + negative control 2: norm-matched RANDOM adapter -> must NOT show a preference shift
"""
import argparse, datetime, hashlib, json, math, os
import numpy as np, torch, transformers, tokenizers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID, MODEL_REV = "Qwen/Qwen2.5-1.5B-Instruct", "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
EVAL_PATH = "data/eval.jsonl"
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_R, LORA_ALPHA = 16, 32
SOUP_PROMPT_CAP, SOUP_TOTAL_CAP = 512, 1024      # Soup 0.75.1 cap_dpo: max_length//2, max_length
ENCODING = "trl-0.29.1-nonconv: eos appended to text; tok(prompt); tok(prompt+completion); suffix slice"
# ---- Pre-registered thresholds (fixed BEFORE training; do not tune after seeing results) ----
NOISE_ABS_TOL = 1e-2          # nats; per-pair |Δ| below this = numerical noise
N_BOOT, BOOT_SEED = 10000, 0  # bootstrap CI for mean Δ
DEVICE = "cuda"

def sha256(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat()

def load_base():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REV, dtype=torch.float16).to(DEVICE)
    model.eval()
    return tok, model

def ids(tok, text):
    return tok(text)["input_ids"]

def encode(tok, rows):
    """Reproduce TRL 0.29.1 non-conversational preprocessing. Returns (enc, encode_stats)."""
    eos = tok.eos_token
    # Tokenizer-call semantics check: default call vs add_special_tokens=False must agree for this tokenizer.
    probe = rows[0]["prompt"] + rows[0]["chosen"]
    special_tokens_agree = ids(tok, probe) == tok(probe, add_special_tokens=False)["input_ids"]
    assert special_tokens_agree, "tokenizer adds special tokens by default; confirm TRL's exact call"
    enc, boundary_diff = [], 0
    for i, r in enumerate(rows):
        p_ids = ids(tok, r["prompt"])
        comps = []
        for field in ("chosen", "rejected"):
            text = r[field] if r[field].endswith(eos) else r[field] + eos
            full = ids(tok, r["prompt"] + text)
            assert full[:len(p_ids)] == p_ids, f"row {i} {field}: full sequence does not start with prompt ids"
            c_ids = full[len(p_ids):]
            assert len(c_ids) >= 1 and c_ids[-1] == tok.eos_token_id, f"row {i} {field}: completion must end with EOS"
            sep = tok(r[field], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
            boundary_diff += int(sep != c_ids)
            comps.append(c_ids)
        c_ids, j_ids = comps
        assert len(p_ids) >= 1
        assert len(p_ids) <= SOUP_PROMPT_CAP, f"row {i}: prompt {len(p_ids)} > {SOUP_PROMPT_CAP} (Soup would truncate)"
        assert len(p_ids) + max(len(c_ids), len(j_ids)) <= SOUP_TOTAL_CAP, f"row {i}: total > {SOUP_TOTAL_CAP} (Soup would truncate)"
        enc.append((p_ids, c_ids, j_ids))
    stats = {"special_tokens_default_equals_false": special_tokens_agree,
             "completions_where_separate_tokenization_differs": boundary_diff,
             "completions_total": 2 * len(rows),
             "max_prompt_tok": max(len(p) for p, _, _ in enc),
             "max_total_tok": max(len(p) + max(len(c), len(j)) for p, c, j in enc),
             "soup_cap_noop": True}
    return enc, stats

@torch.inference_mode()
def completion_logp(model, p_ids, c_ids):
    x = torch.tensor([p_ids + c_ids], device=DEVICE)
    logits = model(input_ids=x, use_cache=False).logits          # [1, P+C, V], fp16
    P, C = len(p_ids), len(c_ids)
    lg = logits[0, P - 1:P + C - 1].float()                      # logits at t predict token t+1
    tgt = x[0, P:P + C]
    lp = torch.log_softmax(lg, dim=-1).gather(-1, tgt[:, None]).squeeze(-1)
    assert lp.shape[0] == C and torch.isfinite(lp).all(), "bad logp slice / non-finite"
    return float(lp.sum()), C

def score(model, enc):
    out = []
    for i, (p, c, j) in enumerate(enc):
        lc, nc = completion_logp(model, p, c)
        lr, nr = completion_logp(model, p, j)
        out.append({"i": i, "prompt_tok": len(p), "chosen_tok": nc, "rejected_tok": nr,
                    "logp_chosen": lc, "logp_rejected": lr, "margin": lc - lr,
                    "margin_per_tok": lc / nc - lr / nr})
    return out

def summarize(rows):
    m = np.array([r["margin"] for r in rows])
    return {"n": len(rows),
            "mean_logp_chosen": float(np.mean([r["logp_chosen"] for r in rows])),
            "mean_logp_rejected": float(np.mean([r["logp_rejected"] for r in rows])),
            "mean_margin": float(m.mean()), "median_margin": float(np.median(m)),
            "frac_chosen_gt_rejected": float((m > 0).mean()),
            "mean_margin_per_tok": float(np.mean([r["margin_per_tok"] for r in rows]))}

def compare(base_rows, new_rows):
    """Δ_i = new margin - base margin = (implicit DPO reward margin)/beta, since ref = base (lora_B=0 init)."""
    d = np.array([n["margin"] - b["margin"] for b, n in zip(base_rows, new_rows)])
    rng = np.random.default_rng(BOOT_SEED)
    boots = d[rng.integers(0, len(d), (N_BOOT, len(d)))].mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    nz = d[np.abs(d) > NOISE_ABS_TOL]; k = int((nz > 0).sum()); n = len(nz)
    p_sign = sum(math.comb(n, i) for i in range(k, n + 1)) / 2 ** n if n else 1.0   # one-sided exact
    len_diff = np.array([b["chosen_tok"] - b["rejected_tok"] for b in base_rows], dtype=float)
    corr = float(np.corrcoef(d, len_diff)[0, 1]) if d.std() > 0 and len_diff.std() > 0 else None
    return {"mean_delta": float(d.mean()), "ci95_mean_delta": [float(lo), float(hi)],
            "frac_delta_pos": float((d > NOISE_ABS_TOL).mean()), "frac_delta_neg": float((d < -NOISE_ABS_TOL).mean()),
            "max_abs_delta": float(np.abs(d).max()), "sign_test_p_one_sided": p_sign,
            "corr_delta_vs_len_diff": corr, "deltas": d.tolist()}

def behaviour_shift(cmp):
    return cmp["ci95_mean_delta"][0] > 0 and cmp["mean_delta"] > NOISE_ABS_TOL

def lora_tensors(model):
    return {n: p.detach().float() for n, p in model.named_parameters() if "lora_" in n}

def classify(param_ok, cmp):
    if not param_ok: return "NOT_TRAINED (adapter params absent/zero/not loaded)"
    if cmp["max_abs_delta"] <= NOISE_ABS_TOL: return "NOT_TRAINED (params non-zero but no measurable effect)"
    if behaviour_shift(cmp): return "TRAINED (params changed AND held-out preference margin increased)"
    return "PARAMS_CHANGED_ONLY (outputs changed, no held-out preference shift)"

def meta(tok, enc_stats, extra=None):
    m = {"timestamp_utc": now(), "model_id": MODEL_ID, "model_revision": MODEL_REV,
         "eval_path": EVAL_PATH, "eval_sha256": sha256(EVAL_PATH),
         "transformers": transformers.__version__, "tokenizers": tokenizers.__version__,
         "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0), "dtype": "float16 (log_softmax fp32)",
         "encoding": ENCODING, "encode_stats": enc_stats, "eos_token": tok.eos_token,
         "score": "sum logp over completion tokens",
         "thresholds": {"NOISE_ABS_TOL": NOISE_ABS_TOL, "N_BOOT": N_BOOT, "BOOT_SEED": BOOT_SEED}}
    m.update(extra or {}); return m

def dump(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    json.dump(obj, open(path, "w"), ensure_ascii=False, indent=2); print("wrote", path)

def print_summary(name, s): print(f"[{name}] " + json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in s.items()}))

def run_baseline(args):
    from peft import LoraConfig, get_peft_model
    tok, model = load_base()
    rows = [json.loads(l) for l in open(EVAL_PATH)]
    enc, est = encode(tok, rows)
    print("[encode] " + json.dumps(est))
    base = score(model, enc)
    rep = score(model, enc[:5])                                   # determinism / noise check
    rep_diff = max(max(abs(a["logp_chosen"] - b["logp_chosen"]), abs(a["logp_rejected"] - b["logp_rejected"])) for a, b in zip(base, rep))
    if rep_diff > NOISE_ABS_TOL: print(f"WARNING: repeat diff {rep_diff:.3e} > NOISE_ABS_TOL")
    s = summarize(base); print_summary("baseline", s); print(f"[baseline] repeat_max_abs_diff={rep_diff:.3e}")
    dump({"meta": meta(tok, est, {"mode": "baseline", "repeat_max_abs_diff": rep_diff}),
          "summary": s, "rows": base}, "artifacts/baseline_logps.json")
    # Negative control 1: zero-initialised LoRA through the same PEFT path.
    pm = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGETS,
                                          lora_dropout=0.0, init_lora_weights=True)).eval()
    lt = lora_tensors(pm); nB = [n for n in lt if "lora_B" in n]
    nB_nonzero = sum(bool(lt[n].abs().sum() > 0) for n in nB)
    ctrl = score(pm, enc); cmp = compare(base, ctrl)
    param_ok = len(nB) > 0 and nB_nonzero == len(nB)
    verdict = classify(param_ok, cmp)
    print(f"[control_zero_adapter] lora_B tensors={len(nB)} nonzero={nB_nonzero} mean_delta={cmp['mean_delta']:.3e} "
          f"max_abs_delta={cmp['max_abs_delta']:.3e} -> {verdict}")
    print("[control_zero_adapter] EXPECTED: NOT_TRAINED ->", "PASS" if verdict.startswith("NOT_TRAINED") else "FAIL")
    dump({"meta": meta(tok, est, {"mode": "control_zero_adapter"}),
          "lora_B_total": len(nB), "lora_B_nonzero": nB_nonzero, "comparison": cmp, "verdict": verdict,
          "rows": ctrl}, "artifacts/control_zero_adapter.json")

def run_post(args):
    from peft import PeftModel
    from safetensors.torch import load_file
    base_art = json.load(open("artifacts/baseline_logps.json"))
    tok, model = load_base()
    assert base_art["meta"]["eval_sha256"] == sha256(EVAL_PATH), "eval file changed since baseline!"
    assert base_art["meta"]["encoding"] == ENCODING, "encoding path differs from baseline!"
    rows = [json.loads(l) for l in open(EVAL_PATH)]; enc, est = encode(tok, rows)
    # --- parameter evidence straight from the saved file ---
    sd = load_file(os.path.join(args.adapter, "adapter_model.safetensors"))
    keys = list(sd); inner = [k for k in keys if ".inner." in k]
    fB = {k: v.float() for k, v in sd.items() if "lora_B" in k}
    fB_nonzero = sum(bool(v.abs().sum() > 0) for v in fB.values())
    pm = PeftModel.from_pretrained(model, args.adapter).eval()
    live = lora_tensors(pm)
    def live_name(k): return k.replace(".lora_A.weight", ".lora_A.default.weight").replace(".lora_B.weight", ".lora_B.default.weight")
    matched = [k for k in keys if live_name(k) in live]
    equal = [k for k in matched if torch.allclose(live[live_name(k)].cpu(), sd[k].float(), atol=1e-3, rtol=1e-3)]
    norms = {k: float(v.norm()) for k, v in fB.items()}
    param_ok = len(fB) > 0 and fB_nonzero == len(fB) and len(equal) == len(keys) and not inner
    post = score(pm, enc); cmp = compare(base_art["rows"], post); verdict = classify(param_ok, cmp)
    s = summarize(post); print_summary("post", s)
    print(f"[params] file_keys={len(keys)} inner_keys={len(inner)} lora_B={len(fB)} lora_B_nonzero={fB_nonzero} "
          f"loaded_and_equal={len(equal)}/{len(keys)} lora_B_norm[min/median/max]="
          f"{min(norms.values()):.3e}/{float(np.median(list(norms.values()))):.3e}/{max(norms.values()):.3e}")
    print(f"[post vs baseline] mean_delta={cmp['mean_delta']:.4f} ci95={cmp['ci95_mean_delta']} "
          f"frac_pos={cmp['frac_delta_pos']:.2f} frac_neg={cmp['frac_delta_neg']:.2f} sign_p={cmp['sign_test_p_one_sided']:.3g} "
          f"corr(Δ,len_diff)={cmp['corr_delta_vs_len_diff']} -> {verdict}")
    dump({"meta": meta(tok, est, {"mode": "post", "adapter": args.adapter}),
          "params": {"file_keys": len(keys), "inner_keys": inner[:5], "lora_B": len(fB), "lora_B_nonzero": fB_nonzero,
                     "loaded_and_equal": len(equal), "lora_B_norms": norms},
          "summary": s, "comparison": cmp, "verdict": verdict, "rows": post}, "artifacts/post_logps.json")
    # Negative control 2: same per-tensor norms, random direction -> params "changed" but should show NO preference shift.
    g = torch.Generator(device="cpu").manual_seed(0)
    with torch.no_grad():
        for n, p in pm.named_parameters():
            if "lora_B" in n:
                r = torch.randn(p.shape, generator=g)
                p.copy_((r / r.norm() * p.detach().float().norm().cpu()).to(p.dtype).to(p.device))
    ctrl = score(pm, enc); ccmp = compare(base_art["rows"], ctrl); cverdict = classify(True, ccmp)
    print(f"[control_random_adapter] mean_delta={ccmp['mean_delta']:.4f} ci95={ccmp['ci95_mean_delta']} -> {cverdict}")
    print("[control_random_adapter] EXPECTED: not TRAINED ->", "PASS" if not cverdict.startswith("TRAINED") else "FAIL")
    dump({"meta": meta(tok, est, {"mode": "control_random_adapter", "seed": 0}),
          "comparison": ccmp, "verdict": cverdict, "rows": ctrl}, "artifacts/control_random_adapter.json")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--baseline", action="store_true"); g.add_argument("--adapter", type=str)
    a = ap.parse_args()
    run_baseline(a) if a.baseline else run_post(a)

Overwriting verify_training.py


In [ ]:
%%bash

mkdir -p artifacts logs/verify

(
  date -u
  /content/venv312/bin/python verify_training.py --baseline
) 2>&1 | tee logs/verify/baseline.log

Wed Sep 30 12:56:42 PM UTC 2026
Loading weights: 100%|██████████| 338/338 [00:09<00:00, 34.34it/s]
[encode] {"special_tokens_default_equals_false": true, "completions_where_separate_tokenization_differs": 0, "completions_total": 100, "max_prompt_tok": 457, "max_total_tok": 872, "soup_cap_noop": true}
[baseline] {"n": 50, "mean_logp_chosen": -234.9394, "mean_logp_rejected": -250.916, "mean_margin": 15.9766, "median_margin": -5.7593, "frac_chosen_gt_rejected": 0.46, "mean_margin_per_tok": 0.1779}
[baseline] repeat_max_abs_diff=0.000e+00
wrote artifacts/baseline_logps.json
[control_zero_adapter] lora_B tensors=196 nonzero=0 mean_delta=0.000e+00 max_abs_delta=0.000e+00 -> NOT_TRAINED (adapter params absent/zero/not loaded)
[control_zero_adapter] EXPECTED: NOT_TRAINED -> PASS
wrote artifacts/control_zero_adapter.json


In [ ]:
%%bash

(
  date -u
  sha256sum \
    verify_training.py \
    artifacts/baseline_logps.json \
    artifacts/control_zero_adapter.json \
    data/eval.jsonl
) | tee logs/verify/pretrain_sha256.txt

Wed Sep 30 01:00:27 PM UTC 2026
f44ce1a3e7f52e280bb1f9add56c15597fe99d46bba008ec8da932a63d89b822  verify_training.py
35678590acd756987c29ce1456097dd764cd5f937a6601d6598f4b5c0a6b456f  artifacts/baseline_logps.json
217fb4c58bf771bf7cac9199518260c9f07b2a94799d78f095ef2fcb0193974d  artifacts/control_zero_adapter.json
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/eval.jsonl


In [ ]:
%%bash
set -o pipefail

mkdir -p logs/train

# Зафиксировать точные входы ДО training
(
  date -u
  sha256sum soup.yaml data/train.jsonl data/eval.jsonl verify_training.py
) | tee logs/train/pretrain_inputs_sha256.txt

# GPU непосредственно перед запуском
(date -u; nvidia-smi) \
  2>&1 | tee logs/train/nvidia_smi_before.txt

# Каждую секунду пишем использование VRAM/GPU
nvidia-smi \
  --query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu \
  --format=csv \
  -l 1 \
  > logs/train/nvidia_smi_sampling.csv 2>&1 &

MONITOR_PID=$!

echo "=== TRAIN START $(date -u) ===" | tee logs/train/train.log

/content/venv312/bin/soup --log-level verbose train -c soup.yaml \
  2>&1 | tee -a logs/train/train.log

TRAIN_EXIT=${PIPESTATUS[0]}

echo "=== TRAIN END $(date -u) | exit=$TRAIN_EXIT ===" \
  | tee -a logs/train/train.log

kill "$MONITOR_PID" 2>/dev/null || true
wait "$MONITOR_PID" 2>/dev/null || true

(date -u; nvidia-smi) \
  2>&1 | tee logs/train/nvidia_smi_after.txt

exit "$TRAIN_EXIT"

Wed Sep 30 01:01:56 PM UTC 2026
a005fad666718f445f1cea8b3bc812d8f14b9b9c5e255c45df76331217add576  soup.yaml
53dee71a59fed7a8103c59fc6e6cfb58d9016c6a74d7b89e5a74710490477027  data/train.jsonl
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/eval.jsonl
f44ce1a3e7f52e280bb1f9add56c15597fe99d46bba008ec8da932a63d89b822  verify_training.py
Wed Sep 30 01:01:56 PM UTC 2026
Wed Sep 30 13:01:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================

CalledProcessError: Command 'b'set -o pipefail\n\nmkdir -p logs/train\n\n# \xd0\x97\xd0\xb0\xd1\x84\xd0\xb8\xd0\xba\xd1\x81\xd0\xb8\xd1\x80\xd0\xbe\xd0\xb2\xd0\xb0\xd1\x82\xd1\x8c \xd1\x82\xd0\xbe\xd1\x87\xd0\xbd\xd1\x8b\xd0\xb5 \xd0\xb2\xd1\x85\xd0\xbe\xd0\xb4\xd1\x8b \xd0\x94\xd0\x9e training\n(\n  date -u\n  sha256sum soup.yaml data/train.jsonl data/eval.jsonl verify_training.py\n) | tee logs/train/pretrain_inputs_sha256.txt\n\n# GPU \xd0\xbd\xd0\xb5\xd0\xbf\xd0\xbe\xd1\x81\xd1\x80\xd0\xb5\xd0\xb4\xd1\x81\xd1\x82\xd0\xb2\xd0\xb5\xd0\xbd\xd0\xbd\xd0\xbe \xd0\xbf\xd0\xb5\xd1\x80\xd0\xb5\xd0\xb4 \xd0\xb7\xd0\xb0\xd0\xbf\xd1\x83\xd1\x81\xd0\xba\xd0\xbe\xd0\xbc\n(date -u; nvidia-smi) \\\n  2>&1 | tee logs/train/nvidia_smi_before.txt\n\n# \xd0\x9a\xd0\xb0\xd0\xb6\xd0\xb4\xd1\x83\xd1\x8e \xd1\x81\xd0\xb5\xd0\xba\xd1\x83\xd0\xbd\xd0\xb4\xd1\x83 \xd0\xbf\xd0\xb8\xd1\x88\xd0\xb5\xd0\xbc \xd0\xb8\xd1\x81\xd0\xbf\xd0\xbe\xd0\xbb\xd1\x8c\xd0\xb7\xd0\xbe\xd0\xb2\xd0\xb0\xd0\xbd\xd0\xb8\xd0\xb5 VR

In [ ]:
%%bash

mkdir -p logs/train

/content/venv312/bin/python - <<'PY' 2>&1 | tee logs/train/hf_cache_diagnostic.txt
import os
import huggingface_hub
from huggingface_hub import snapshot_download

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
REV = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

snap = snapshot_download(
    MODEL,
    revision=REV,
    local_files_only=True,
)

p = os.path.join(snap, "model.safetensors")
repo_root = os.path.dirname(os.path.dirname(snap))
blob_root = os.path.realpath(os.path.join(repo_root, "blobs"))
real = os.path.realpath(p)

print("huggingface_hub:", huggingface_hub.__version__)
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))
print()
print("snapshot:", snap)
print("model.safetensors exists:", os.path.exists(p))
print("is symlink:", os.path.islink(p))
print("readlink:", os.readlink(p) if os.path.islink(p) else None)
print("realpath:", real)
print("expected blob root:", blob_root)
print(
    "inside expected blob root:",
    os.path.commonpath([real, blob_root]) == blob_root
)
PY

Traceback (most recent call last):
  File "<stdin>", line 8, in <module>
  File "/content/venv312/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py", line 89, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/content/venv312/lib/python3.12/site-packages/huggingface_hub/utils/_http.py", line 266, in _inner
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/content/venv312/lib/python3.12/site-packages/huggingface_hub/_snapshot_download.py", line 330, in snapshot_download
    _raise_if_incomplete_snapshot(
  File "/content/venv312/lib/python3.12/site-packages/huggingface_hub/_snapshot_download.py", line 586, in _raise_if_incomplete_snapshot
    raise IncompleteSnapshotError(
huggingface_hub.errors.IncompleteSnapshotError: The cached snapshot for 'Qwen/Qwen2.5-1.5B-Instruct' (revision '989aa7980e4cf806f80c7fef2b1adb7bc71aa306', commit 989aa7980e4cf806f80c7fef2b1adb7bc71aa306) is incomplete: 3 file(s) are missing (.gitatt

In [ ]:
%%bash

/content/venv312/bin/python - <<'PY' 2>&1 | tee logs/train/hf_cache_diagnostic_v2.txt
import os
from huggingface_hub.constants import HF_HUB_CACHE

REV = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

repo = os.path.join(
    HF_HUB_CACHE,
    "models--Qwen--Qwen2.5-1.5B-Instruct"
)
snap = os.path.join(repo, "snapshots", REV)
blob_root = os.path.realpath(os.path.join(repo, "blobs"))

print("HF_HUB_CACHE:", HF_HUB_CACHE)
print("repo:", repo)
print("snapshot:", snap)
print("snapshot exists:", os.path.isdir(snap))
print("expected blob root:", blob_root)
print()

if os.path.isdir(snap):
    print("snapshot files:")
    for name in sorted(os.listdir(snap)):
        p = os.path.join(snap, name)
        print(
            f"{name:35} "
            f"symlink={os.path.islink(p)} "
            f"exists={os.path.exists(p)}"
        )

    print("\nmodel.safetensors detail:")
    p = os.path.join(snap, "model.safetensors")
    print("path:", p)
    print("is symlink:", os.path.islink(p))
    if os.path.islink(p):
        print("readlink:", os.readlink(p))
    print("realpath:", os.path.realpath(p))
    print(
        "inside expected blob root:",
        os.path.commonpath(
            [os.path.realpath(p), blob_root]
        ) == blob_root
    )
PY

HF_HUB_CACHE: /root/.cache/huggingface/hub
repo: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct
snapshot: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306
snapshot exists: True
expected blob root: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/blobs

snapshot files:
config.json                         symlink=True exists=True
generation_config.json              symlink=True exists=True
merges.txt                          symlink=True exists=True
model.safetensors                   symlink=True exists=True
tokenizer.json                      symlink=True exists=True
tokenizer_config.json               symlink=True exists=True
vocab.json                          symlink=True exists=True

model.safetensors detail:
path: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/model.safetensors
is symlink: True
readlink: ../../blo

In [ ]:
%%bash
set -euo pipefail

SRC="/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
DST="/content/models/qwen2.5-1.5b-pinned"

rm -rf "$DST"
mkdir -p "$DST" logs/train

for f in \
  config.json \
  generation_config.json \
  merges.txt \
  model.safetensors \
  tokenizer.json \
  tokenizer_config.json \
  vocab.json
do
  cp -L "$SRC/$f" "$DST/$f"
done

(
  date -u
  echo "=== LOCAL MODEL ==="
  ls -lh "$DST"

  echo
  echo "=== SYMLINKS (should be empty) ==="
  find "$DST" -type l -print

  echo
  echo "=== WEIGHT HASH COMPARISON ==="
  sha256sum "$SRC/model.safetensors"
  sha256sum "$DST/model.safetensors"
) | tee logs/train/local_model_materialization.txt

Wed Sep 30 01:12:50 PM UTC 2026
=== LOCAL MODEL ===
total 2.9G
-rw-r--r-- 1 root root  660 Sep 30 13:12 config.json
-rw-r--r-- 1 root root  242 Sep 30 13:12 generation_config.json
-rw-r--r-- 1 root root 1.6M Sep 30 13:12 merges.txt
-r--r--r-- 1 root root 2.9G Sep 30 13:12 model.safetensors
-rw-r--r-- 1 root root 7.2K Sep 30 13:12 tokenizer_config.json
-rw-r--r-- 1 root root 6.8M Sep 30 13:12 tokenizer.json
-rw-r--r-- 1 root root 2.7M Sep 30 13:12 vocab.json

=== SYMLINKS (should be empty) ===

=== WEIGHT HASH COMPARISON ===
dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee  /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/model.safetensors
dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee  /content/models/qwen2.5-1.5b-pinned/model.safetensors


In [ ]:
%%bash
set -e

cp soup.yaml soup_before_local_base.yaml

python - <<'PY'
from pathlib import Path

p = Path("soup.yaml")
text = p.read_text()

old = "base: Qwen/Qwen2.5-1.5B-Instruct"
new = "base: /content/models/qwen2.5-1.5b-pinned"

assert old in text, "Original base line not found"
text = text.replace(old, new, 1)
p.write_text(text)

print("UPDATED:")
for line in text.splitlines():
    if line.startswith("base:"):
        print(line)
PY

echo
echo "=== DIFF ==="
diff -u soup_before_local_base.yaml soup.yaml || true

echo
echo "=== NEW CONFIG HASH ==="
sha256sum soup.yaml

UPDATED:
base: /content/models/qwen2.5-1.5b-pinned

=== DIFF ===
--- soup_before_local_base.yaml	2026-09-30 13:16:01.493737758 +0000
+++ soup.yaml	2026-09-30 13:16:01.533741753 +0000
@@ -1,5 +1,5 @@
 
-base: Qwen/Qwen2.5-1.5B-Instruct
+base: /content/models/qwen2.5-1.5b-pinned
 task: dpo
 backend: transformers
 

=== NEW CONFIG HASH ===
82ea3a7ca287bb51272f667aa1d2cdb84ccd6bdbff96d1bf8e1390da447767e3  soup.yaml


In [ ]:
%%bash

mkdir -p logs/preflight_local_base

(
  date -u
  /content/venv312/bin/soup --log-level verbose train -c soup.yaml --dry-run
) 2>&1 | tee logs/preflight_local_base/dry_run.txt

Wed Sep 30 01:18:21 PM UTC 2026
Loading config from soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   /content/models/qwen2.5-1.5b-pinned                                 │
│ Task:    dpo                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=16, alpha=32                                                      │
│ Quant:   none                                                                │
│ Seed:    42                                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
Dry run - validating data...
Data OK: 500 train samples
Config valid. Ready to train!


In [ ]:
%%bash
set -o pipefail

mkdir -p logs/train_attempt2

# Точные входы второго запуска
(
  date -u
  sha256sum soup.yaml data/train.jsonl data/eval.jsonl verify_training.py
  sha256sum /content/models/qwen2.5-1.5b-pinned/model.safetensors
) | tee logs/train_attempt2/pretrain_inputs_sha256.txt

# GPU до training
(date -u; nvidia-smi) \
  2>&1 | tee logs/train_attempt2/nvidia_smi_before.txt

# Монитор VRAM каждую секунду
nvidia-smi \
  --query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu \
  --format=csv \
  -l 1 \
  > logs/train_attempt2/nvidia_smi_sampling.csv 2>&1 &

MONITOR_PID=$!

echo "=== TRAIN ATTEMPT 2 START $(date -u) ===" \
  | tee logs/train_attempt2/train.log

/content/venv312/bin/soup --log-level verbose train -c soup.yaml \
  2>&1 | tee -a logs/train_attempt2/train.log

TRAIN_EXIT=${PIPESTATUS[0]}

echo "=== TRAIN ATTEMPT 2 END $(date -u) | exit=$TRAIN_EXIT ===" \
  | tee -a logs/train_attempt2/train.log

kill "$MONITOR_PID" 2>/dev/null || true
wait "$MONITOR_PID" 2>/dev/null || true

(date -u; nvidia-smi) \
  2>&1 | tee logs/train_attempt2/nvidia_smi_after.txt

exit "$TRAIN_EXIT"

Wed Sep 30 01:20:05 PM UTC 2026
82ea3a7ca287bb51272f667aa1d2cdb84ccd6bdbff96d1bf8e1390da447767e3  soup.yaml
53dee71a59fed7a8103c59fc6e6cfb58d9016c6a74d7b89e5a74710490477027  data/train.jsonl
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/eval.jsonl
f44ce1a3e7f52e280bb1f9add56c15597fe99d46bba008ec8da932a63d89b822  verify_training.py
dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee  /content/models/qwen2.5-1.5b-pinned/model.safetensors
Wed Sep 30 01:20:22 PM UTC 2026
Wed Sep 30 13:20:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. 

In [ ]:
%%bash

/content/venv312/bin/python - <<'PY' 2>&1 | tee logs/train_attempt2/token_boundary_audit.txt
import json
from transformers import AutoTokenizer

MODEL = "/content/models/qwen2.5-1.5b-pinned"
tok = AutoTokenizer.from_pretrained(MODEL)
eos = tok.eos_token

rows = [json.loads(x) for x in open("data/train.jsonl")]

mismatches = []

for i, r in enumerate(rows):
    p = tok(r["prompt"])["input_ids"]

    for field in ("chosen", "rejected"):
        completion = r[field]
        if not completion.endswith(eos):
            completion += eos

        full = tok(r["prompt"] + completion)["input_ids"]

        if full[:len(p)] != p:
            # first differing token position
            n = min(len(p), len(full))
            first_diff = next(
                (j for j in range(n) if p[j] != full[j]),
                n
            )

            mismatches.append({
                "row": i,
                "field": field,
                "prompt_tokens": len(p),
                "full_tokens": len(full),
                "first_diff_token": first_diff,
            })

print("completion mismatches:", len(mismatches))
print("affected rows:", len(set(x["row"] for x in mismatches)))
print()

for x in mismatches:
    print(x)
PY

completion mismatches: 5
affected rows: 4

{'row': 137, 'field': 'chosen', 'prompt_tokens': 21, 'full_tokens': 179, 'first_diff_token': 20}
{'row': 137, 'field': 'rejected', 'prompt_tokens': 21, 'full_tokens': 99, 'first_diff_token': 20}
{'row': 385, 'field': 'chosen', 'prompt_tokens': 294, 'full_tokens': 390, 'first_diff_token': 293}
{'row': 403, 'field': 'chosen', 'prompt_tokens': 27, 'full_tokens': 216, 'first_diff_token': 26}
{'row': 405, 'field': 'chosen', 'prompt_tokens': 472, 'full_tokens': 544, 'first_diff_token': 471}


In [ ]:
%%bash

mkdir -p data/superseded_v2

cp data/train.jsonl data/superseded_v2/train.jsonl
cp data/eval.jsonl data/superseded_v2/eval.jsonl
cp data/manifest.json data/superseded_v2/manifest.json

sha256sum \
  data/superseded_v2/train.jsonl \
  data/superseded_v2/eval.jsonl \
  data/superseded_v2/manifest.json \
  > data/superseded_v2/SHA256SUMS.txt

cat data/superseded_v2/SHA256SUMS.txt

53dee71a59fed7a8103c59fc6e6cfb58d9016c6a74d7b89e5a74710490477027  data/superseded_v2/train.jsonl
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/superseded_v2/eval.jsonl
e1e73e4b40169aa923daefc93908ba1cf2f93d5071439cbea3e25323cecea1e0  data/superseded_v2/manifest.json


In [ ]:
!grep -n -E "trl_boundary_mismatch|R9_boundary" prepare_data.py

69:def trl_boundary_mismatch(prompt, completion):
287:    if trl_boundary_mismatch(p, c) or trl_boundary_mismatch(p, r):
288:      drops["R9_boundary_tokenization_mismatch"] += 1
569:    "R9_boundary_tokenization_mismatch",


In [ ]:
%%bash

(date -u; /content/venv312/bin/python prepare_data.py) \
  2>&1 | tee logs/data_v3_prepare.txt

Wed Sep 30 01:51:14 PM UTC 2026
Loading dataset...
source rows: 112052
Traceback (most recent call last):
  File "/content/prepare_data.py", line 287, in <module>
    if trl_boundary_mismatch(p, c) or trl_boundary_mismatch(p, r):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/prepare_data.py", line 71, in trl_boundary_mismatch
    assert tok.eos_token is not None
           ^^^
NameError: name 'tok' is not defined. Did you mean: 'ntok'?


In [ ]:
!grep -n -E "AutoTokenizer|def ntok" prepare_data.py

12:from transformers import AutoTokenizer
39:TOK = AutoTokenizer.from_pretrained(
57:def ntok(text):


In [ ]:
from pathlib import Path

p = Path("prepare_data.py")
text = p.read_text()

text = text.replace("tok.eos_token", "TOK.eos_token")
text = text.replace("tok(prompt)", "TOK(prompt)")
text = text.replace("tok(prompt + completion_with_eos)", "TOK(prompt + completion_with_eos)")

p.write_text(text)

print("saved")

saved


In [ ]:
!sed -n '65,82p' prepare_data.py


# ============================================================
# Helpers
# ============================================================
def trl_boundary_mismatch(prompt, completion):
    """Match TRL 0.29.1 raw-text DPO preprocessing at the prompt/completion boundary."""
    assert TOK.eos_token is not None

    completion_with_eos = (
        completion
        if completion.endswith(TOK.eos_token)
        else completion + TOK.eos_token
    )

    prompt_ids = TOK(prompt)["input_ids"]
    full_ids = TOK(prompt + completion_with_eos)["input_ids"]

    return full_ids[:len(prompt_ids)] != prompt_ids


In [ ]:
%%bash

(date -u; /content/venv312/bin/python prepare_data.py) \
  2>&1 | tee logs/data_v3_prepare.txt

Wed Sep 30 01:56:26 PM UTC 2026
Loading dataset...
source rows: 112052
{
  "repo": "d0rj/full-hh-rlhf-ru",
  "revision": "fd4a069df1b99329527389101f48540140478ff7",
  "source_split": "train",
  "seed": 42,
  "rows_scanned": 984,
  "drops": {
    "R1_empty": 1,
    "R2_identical": 6,
    "R3_low_cyrillic": 328,
    "R4_over_3000_chars": 37,
    "R5_dup_exact_prompt": 0,
    "R6_completion_overlap_with_eval": 4,
    "R7_prompt_over_500_tok": 54,
    "R8_total_over_1000_tok": 0,
    "R9_boundary_tokenization_mismatch": 4
  },
  "drop_counter_scope": "Drop counts apply only to rows scanned until 50 eval + 500 train rows were collected; they are not full-dataset prevalence estimates.",
  "selection": {
    "eval": "first 50 eligible rows after deterministic shuffle",
    "train": "next 500 eligible rows after deterministic shuffle; train completions are excluded if they exactly overlap eval completions"
  },
  "filters": {
    "MAX_CHARS": 3000,
    "MIN_CYR": 0.5
  },
  "tokenizer": {
    

In [ ]:
%%bash
/content/venv312/bin/python - <<'PY' 2>&1 | tee logs/data_v3_boundary_audit.txt
import json
from transformers import AutoTokenizer

MODEL = "/content/models/qwen2.5-1.5b-pinned"
tok = AutoTokenizer.from_pretrained(MODEL)
eos = tok.eos_token

def audit(path):
    rows = [json.loads(x) for x in open(path)]
    bad = []

    for i, r in enumerate(rows):
        p = tok(r["prompt"])["input_ids"]

        for field in ("chosen", "rejected"):
            c = r[field]
            if not c.endswith(eos):
                c += eos

            full = tok(r["prompt"] + c)["input_ids"]

            if full[:len(p)] != p:
                bad.append((i, field))

    return len(rows), bad

n_train, bad_train = audit("data/train.jsonl")
n_eval, bad_eval = audit("data/eval.jsonl")

print("train rows:", n_train)
print("train boundary mismatches:", len(bad_train), bad_train)
print("eval rows:", n_eval)
print("eval boundary mismatches:", len(bad_eval), bad_eval)

assert not bad_train
assert not bad_eval

print("\nBOUNDARY AUDIT PASS")
PY

train rows: 500
train boundary mismatches: 0 []
eval rows: 50
eval boundary mismatches: 0 []

BOUNDARY AUDIT PASS


In [ ]:
%%bash
set -e

echo "=== FILE HASHES ==="
sha256sum data/train.jsonl data/eval.jsonl soup.yaml

echo
echo "=== ROW COUNTS ==="
/content/venv312/bin/python - <<'PY'
import json
train = [json.loads(x) for x in open("data/train.jsonl")]
ev = [json.loads(x) for x in open("data/eval.jsonl")]
print("train:", len(train))
print("eval :", len(ev))
assert len(train) == 500
assert len(ev) == 50
PY

echo
echo "=== SOUP VALIDATE ==="
/content/venv312/bin/soup data validate data/train.jsonl --format dpo

echo
echo "=== SOUP LINT ==="
/content/venv312/bin/soup data lint data/train.jsonl --format dpo

echo
echo "=== CONFIG DRY RUN ==="
/content/venv312/bin/soup train soup.yaml --dry-run

echo
echo "FINAL PREFLIGHT PASS"

=== FILE HASHES ===
175f3ad1ae66034c557f7d54efe30121c6e5b7e99169127539dacb2207e3df67  data/train.jsonl
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/eval.jsonl
82ea3a7ca287bb51272f667aa1d2cdb84ccd6bdbff96d1bf8e1390da447767e3  soup.yaml

=== ROW COUNTS ===
train: 500
eval : 50

=== SOUP VALIDATE ===
Dataset is valid!

500/500 rows valid for dpo format

=== SOUP LINT ===
                              soup data lint (dpo)                              
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Check           ┃ Verdict ┃ Message                ┃ Evidence                ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ length_bias     │ OK      │ effect size (Cohen's   │ mean chosen=54.4, mean  │
│                 │         │ d) = 0.106 (chosen     │ rejected=49.9           │
│                 │         │ longer than rejected); │                         │
│                 │         │ chosen

Usage: soup train [OPTIONS]
Try 'soup train --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Got unexpected extra argument (soup.yaml)                                    │
╰──────────────────────────────────────────────────────────────────────────────╯


CalledProcessError: Command 'b'set -e\n\necho "=== FILE HASHES ==="\nsha256sum data/train.jsonl data/eval.jsonl soup.yaml\n\necho\necho "=== ROW COUNTS ==="\n/content/venv312/bin/python - <<\'PY\'\nimport json\ntrain = [json.loads(x) for x in open("data/train.jsonl")]\nev = [json.loads(x) for x in open("data/eval.jsonl")]\nprint("train:", len(train))\nprint("eval :", len(ev))\nassert len(train) == 500\nassert len(ev) == 50\nPY\n\necho\necho "=== SOUP VALIDATE ==="\n/content/venv312/bin/soup data validate data/train.jsonl --format dpo\n\necho\necho "=== SOUP LINT ==="\n/content/venv312/bin/soup data lint data/train.jsonl --format dpo\n\necho\necho "=== CONFIG DRY RUN ==="\n/content/venv312/bin/soup train soup.yaml --dry-run\n\necho\necho "FINAL PREFLIGHT PASS"\n'' returned non-zero exit status 2.

In [ ]:
%%bash
/content/venv312/bin/soup train --dry-run

Loading config from soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   /content/models/qwen2.5-1.5b-pinned                                 │
│ Task:    dpo                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=16, alpha=32                                                      │
│ Quant:   none                                                                │
│ Seed:    42                                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
Dry run - validating data...
Data OK: 500 train samples
Config valid. Ready to train!


In [ ]:
%%bash
set -e

echo "=== BEFORE ==="
ls -ld output/dpo_qwen15b_stream 2>/dev/null || true

if [ -d output/dpo_qwen15b_stream ]; then
    if [ -e output/run1_pre_R9 ]; then
        echo "ERROR: output/run1_pre_R9 already exists — refusing to overwrite it."
        exit 1
    fi

    mv output/dpo_qwen15b_stream output/run1_pre_R9
    echo "Moved old adapter -> output/run1_pre_R9"
else
    echo "No previous output directory found."
fi

echo
echo "=== AFTER ==="
ls -ld output/run1_pre_R9 2>/dev/null || true
ls -ld output/dpo_qwen15b_stream 2>/dev/null || echo "final output path is clean"

echo
echo "OUTPUT ARCHIVE PASS"

=== BEFORE ===
drwxr-xr-x 4 root root 4096 Sep 30 13:30 output/dpo_qwen15b_stream
Moved old adapter -> output/run1_pre_R9

=== AFTER ===
drwxr-xr-x 4 root root 4096 Sep 30 13:30 output/run1_pre_R9
final output path is clean

OUTPUT ARCHIVE PASS


### Historical pre-R9 audit - executed out of final-data order
The following preserved output refers to the superseded v2 training sample (`train SHA 53dee...`), before R9 filtering. It is kept intentionally as evidence of the discovered token-boundary failure. The final v3 dataset has SHA `175f3ad...` and is verified separately by `data_v3_boundary_audit` with 0 train and 0 eval mismatches, then reproduced again after the Colab reset.


In [ ]:
%%bash

/content/venv312/bin/python - <<'PY'
import json, hashlib
from transformers import AutoTokenizer

EXPECTED_EVAL = "ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2"
MODEL = "/content/models/qwen2.5-1.5b-pinned"

def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

train = [json.loads(x) for x in open("data/train.jsonl")]
eval_rows = [json.loads(x) for x in open("data/eval.jsonl")]
manifest = json.load(open("data/manifest.json"))

print("train rows:", len(train))
print("eval rows:", len(eval_rows))
print("train sha256:", sha256("data/train.jsonl"))
print("eval sha256 :", sha256("data/eval.jsonl"))
print("eval unchanged:", sha256("data/eval.jsonl") == EXPECTED_EVAL)
print("R9 drops:", manifest["drops"].get("R9_boundary_tokenization_mismatch"))

tok = AutoTokenizer.from_pretrained(MODEL)
eos = tok.eos_token

def count_mismatches(rows):
    bad = []
    for i, r in enumerate(rows):
        p = tok(r["prompt"])["input_ids"]
        for field in ("chosen", "rejected"):
            c = r[field]
            if not c.endswith(eos):
                c += eos
            full = tok(r["prompt"] + c)["input_ids"]
            if full[:len(p)] != p:
                bad.append((i, field))
    return bad

bad_train = count_mismatches(train)
bad_eval = count_mismatches(eval_rows)

print("train boundary mismatches:", len(bad_train), bad_train[:10])
print("eval boundary mismatches :", len(bad_eval), bad_eval[:10])

assert len(train) == 500
assert len(eval_rows) == 50
assert sha256("data/eval.jsonl") == EXPECTED_EVAL, "EVAL CHANGED — baseline must be redone"
assert not bad_train, "train still contains boundary mismatches"
assert not bad_eval, "eval contains boundary mismatches"

print("\nALL CHECKS PASS")
PY

train rows: 500
eval rows: 50
train sha256: 53dee71a59fed7a8103c59fc6e6cfb58d9016c6a74d7b89e5a74710490477027
eval sha256 : ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2
eval unchanged: True
R9 drops: None
train boundary mismatches: 5 [(137, 'chosen'), (137, 'rejected'), (385, 'chosen'), (403, 'chosen'), (405, 'chosen')]
eval boundary mismatches : 0 []


Traceback (most recent call last):
  File "<stdin>", line 47, in <module>
AssertionError: train still contains boundary mismatches


CalledProcessError: Command 'b'\n/content/venv312/bin/python - <<\'PY\'\nimport json, hashlib\nfrom transformers import AutoTokenizer\n\nEXPECTED_EVAL = "ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2"\nMODEL = "/content/models/qwen2.5-1.5b-pinned"\n\ndef sha256(path):\n    with open(path, "rb") as f:\n        return hashlib.sha256(f.read()).hexdigest()\n\ntrain = [json.loads(x) for x in open("data/train.jsonl")]\neval_rows = [json.loads(x) for x in open("data/eval.jsonl")]\nmanifest = json.load(open("data/manifest.json"))\n\nprint("train rows:", len(train))\nprint("eval rows:", len(eval_rows))\nprint("train sha256:", sha256("data/train.jsonl"))\nprint("eval sha256 :", sha256("data/eval.jsonl"))\nprint("eval unchanged:", sha256("data/eval.jsonl") == EXPECTED_EVAL)\nprint("R9 drops:", manifest["drops"].get("R9_boundary_tokenization_mismatch"))\n\ntok = AutoTokenizer.from_pretrained(MODEL)\neos = tok.eos_token\n\ndef count_mismatches(rows):\n    bad = []\n    for i, r i

In [ ]:
!nvidia-smi

Thu Oct  1 07:32:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!ls -lah /content

total 16K
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 .
drwxr-xr-x 1 root root 4.0K Oct  1 07:24 ..
drwxr-xr-x 4 root root 4.0K Sep  4 13:32 .config
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data


In [4]:
%%bash
cd /content
unzip -o soup_recovery_bundle.zip
ls -lah

Archive:  soup_recovery_bundle.zip
  inflating: Untitled11_saved.ipynb  
  inflating: README_RECOVERY.md      
  inflating: prepare_data.py         
  inflating: verify_training.py      
  inflating: soup.yaml               
  inflating: run_final.sh            
  inflating: materialize_model.py    
total 768K
drwxr-xr-x 1 root root 4.0K Oct  1 07:44 .
drwxr-xr-x 1 root root 4.0K Oct  1 07:24 ..
drwxr-xr-x 4 root root 4.0K Sep  4 13:32 .config
-rw-r--r-- 1 root root 1.3K Oct  1 07:38 materialize_model.py
-rw-r--r-- 1 root root  16K Oct  1 07:38 prepare_data.py
-rw-r--r-- 1 root root 2.5K Oct  1 07:38 README_RECOVERY.md
-rwxr-xr-x 1 root root 1.3K Oct  1 07:38 run_final.sh
drwxr-xr-x 1 root root 4.0K Oct  1 07:43 sample_data
-rw-r--r-- 1 root root 107K Oct  1 07:44 soup_recovery_bundle.zip
-rw-r--r-- 1 root root  556 Oct  1 07:40 soup.yaml
-rw-r--r-- 1 root root 592K Oct  1 07:36 Untitled11_saved.ipynb
-rw-r--r-- 1 root root  14K Oct  1 07:38 verify_training.py


In [5]:
%%bash
set -e

pip install -q uv

uv venv /content/venv312 --python 3.12

uv pip install \
  --python /content/venv312/bin/python \
  "soup-cli[train,data]==0.75.1"

echo
echo "=== VERSIONS ==="
/content/venv312/bin/python --version
/content/venv312/bin/soup version

/content/venv312/bin/python - <<'PY'
import torch, transformers, trl, peft, accelerate
print("torch:", torch.__version__)
print("cuda :", torch.version.cuda)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 51.3 MB/s eta 0:00:00

=== VERSIONS ===
Python 3.12.3
soup v0.75.1
https://github.com/MakazhanAlpamys/Soup
torch: 2.14.1+cu130
cuda : 13.0
transformers: 5.18.0
trl: 0.29.1
peft: 0.21.1
accelerate: 1.15.0


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: venv312
Activate with: source venv312/bin/activate
Using Python 3.12.3 environment at: venv312
Resolved 89 packages in 893ms
Prepared 89 packages in 1m 23s
Installed 89 packages in 653ms
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + bitsandbytes==0.50.2
 + certifi==2026.7.22
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.2
 + cuda-toolkit==13.0.3.0
 + datasets==5.0.1
 + datasketch==2.0.0
 + dill==0.4.1
 + filelock==4.0.8
 + frozenlist==1.8.0
 + fsspec==2026.6.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.33.0
 + idna==3.20
 + jinja2==3.1.6
 + joblib==1.6.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + mdurl==0.1.2
 + mpmath==1.3.0
 + multidict==6.9.1
 + multiprocess==0.70.19
 + 

In [6]:
%%bash
/content/venv312/bin/uv pip install \
  --python /content/venv312/bin/python \
  "transformers==5.17.0"

echo "=== CHECK ==="
/content/venv312/bin/python - <<'PY'
import torch, transformers, trl, peft, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
PY

=== CHECK ===
torch: 2.14.1+cu130
transformers: 5.18.0
trl: 0.29.1
peft: 0.21.1
accelerate: 1.15.0


bash: line 1: /content/venv312/bin/uv: No such file or directory


In [7]:
%%bash

uv pip install \
  --python /content/venv312/bin/python \
  "transformers==5.17.0"

echo "=== CHECK ==="
/content/venv312/bin/python - <<'PY'
import torch, transformers, trl, peft, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
PY

=== CHECK ===
torch: 2.14.1+cu130
transformers: 5.17.0
trl: 0.29.1
peft: 0.21.1
accelerate: 1.15.0


Using Python 3.12.3 environment at: venv312
Resolved 26 packages in 157ms
Prepared 1 package in 417ms
Uninstalled 1 package in 146ms
Installed 1 package in 82ms
 - transformers==5.18.0
 + transformers==5.17.0


In [8]:
%%bash
set -e

cd /content
mkdir -p data logs artifacts output

/content/venv312/bin/python prepare_data.py \
  2>&1 | tee logs/data_rebuild_final.txt

echo
echo "=== HASHES ==="
sha256sum data/train.jsonl data/eval.jsonl soup.yaml verify_training.py

Loading dataset...
Generating test split: 100%|██████████| 12451/12451 [00:00<00:00, 46906.18 examples/s]
source rows: 112052
{
  "repo": "d0rj/full-hh-rlhf-ru",
  "revision": "fd4a069df1b99329527389101f48540140478ff7",
  "source_split": "train",
  "seed": 42,
  "rows_scanned": 984,
  "drops": {
    "R1_empty": 1,
    "R2_identical": 6,
    "R3_low_cyrillic": 328,
    "R4_over_3000_chars": 37,
    "R5_dup_exact_prompt": 0,
    "R6_completion_overlap_with_eval": 4,
    "R7_prompt_over_500_tok": 54,
    "R8_total_over_1000_tok": 0,
    "R9_boundary_tokenization_mismatch": 4
  },
  "drop_counter_scope": "Drop counts apply only to rows scanned until 50 eval + 500 train rows were collected; they are not full-dataset prevalence estimates.",
  "selection": {
    "eval": "first 50 eligible rows after deterministic shuffle",
    "train": "next 500 eligible rows after deterministic shuffle; train completions are excluded if they exactly overlap eval completions"
  },
  "filters": {
    "MAX_CHAR

In [9]:
%%bash
cd /content

/content/venv312/bin/python materialize_model.py \
  2>&1 | tee logs/model_materialization_final.txt

Fetching 7 files: 100%|██████████| 7/7 [01:09<00:00,  9.98s/it]
snapshot: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306
local model: /content/models/qwen2.5-1.5b-pinned
weight sha256: dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee
expected     : dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee
LOCAL MODEL PASS


In [10]:
%%bash
cd /content

mkdir -p artifacts logs/verify

/content/venv312/bin/python verify_training.py --baseline \
  2>&1 | tee logs/verify/baseline_final_rebuild.log

echo
echo "=== BASELINE ARTIFACT HASHES ==="
sha256sum \
  artifacts/baseline_logps.json \
  artifacts/control_zero_adapter.json \
  data/eval.jsonl

Loading weights: 100%|██████████| 338/338 [00:12<00:00, 27.21it/s]
[encode] {"special_tokens_default_equals_false": true, "completions_where_separate_tokenization_differs": 0, "completions_total": 100, "max_prompt_tok": 457, "max_total_tok": 872, "soup_cap_noop": true}
[baseline] {"n": 50, "mean_logp_chosen": -234.9394, "mean_logp_rejected": -250.916, "mean_margin": 15.9766, "median_margin": -5.7593, "frac_chosen_gt_rejected": 0.46, "mean_margin_per_tok": 0.1779}
[baseline] repeat_max_abs_diff=0.000e+00
wrote artifacts/baseline_logps.json
[control_zero_adapter] lora_B tensors=196 nonzero=0 mean_delta=0.000e+00 max_abs_delta=0.000e+00 -> NOT_TRAINED (adapter params absent/zero/not loaded)
[control_zero_adapter] EXPECTED: NOT_TRAINED -> PASS
wrote artifacts/control_zero_adapter.json

=== BASELINE ARTIFACT HASHES ===
6f70b49a0f8c4442b6811567d87fc080aa6659eefd4d7a686ed73ad5a740dba2  artifacts/baseline_logps.json
70319be1621e64d586d03c7613604d179aae6a9c54689219929725f97486af58  artifacts/co

In [11]:
%%bash
cd /content
set -e

if [ -e output/dpo_qwen15b_stream ]; then
    echo "ERROR: final output path already exists"
    ls -lah output/dpo_qwen15b_stream
    exit 1
fi

echo "FINAL OUTPUT PATH CLEAN"
bash run_final.sh

FINAL OUTPUT PATH CLEAN
=== FINAL RUN START ===
Thu Oct  1 08:02:00 AM UTC 2026

=== INPUT HASHES ===
82ea3a7ca287bb51272f667aa1d2cdb84ccd6bdbff96d1bf8e1390da447767e3  soup.yaml
175f3ad1ae66034c557f7d54efe30121c6e5b7e99169127539dacb2207e3df67  data/train.jsonl
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/eval.jsonl
981595fe71f26feb321efe2bf9af8cdbc76365141daeb1f52979c85920b38553  prepare_data.py
f44ce1a3e7f52e280bb1f9add56c15597fe99d46bba008ec8da932a63d89b822  verify_training.py
dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee  /content/models/qwen2.5-1.5b-pinned/model.safetensors

=== GPU BEFORE ===
Thu Oct  1 08:02:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M 

CalledProcessError: Command 'b'cd /content\nset -e\n\nif [ -e output/dpo_qwen15b_stream ]; then\n    echo "ERROR: final output path already exists"\n    ls -lah output/dpo_qwen15b_stream\n    exit 1\nfi\n\necho "FINAL OUTPUT PATH CLEAN"\nbash run_final.sh\n'' returned non-zero exit status 1.

In [12]:
%%bash
cd /content
set -e

# Preserve the aborted run logs
if [ -d logs/train_final_v3 ]; then
    mv logs/train_final_v3 logs/train_final_v3_aborted_prompt
fi

# Add automatic confirmation to the training command
python - <<'PY'
from pathlib import Path

p = Path("run_final.sh")
s = p.read_text()

old = '/content/venv312/bin/soup --log-level verbose train -c soup.yaml \\'
new = '/content/venv312/bin/soup --log-level verbose train -c soup.yaml -y \\'

assert old in s, "Expected training command not found"
p.write_text(s.replace(old, new, 1))
print("PATCHED run_final.sh")
PY

grep -n "soup .*train" run_final.sh

# Safety: no adapter should exist because training never started
if [ -e output/dpo_qwen15b_stream ]; then
    echo "ERROR: output path unexpectedly exists"
    ls -lah output/dpo_qwen15b_stream
    exit 1
fi

echo "FINAL OUTPUT PATH CLEAN"
echo "STARTING REAL FINAL RUN"

bash run_final.sh

PATCHED run_final.sh
36:/content/venv312/bin/soup --log-level verbose train -c soup.yaml -y \
FINAL OUTPUT PATH CLEAN
STARTING REAL FINAL RUN
=== FINAL RUN START ===
Thu Oct  1 08:03:38 AM UTC 2026

=== INPUT HASHES ===
82ea3a7ca287bb51272f667aa1d2cdb84ccd6bdbff96d1bf8e1390da447767e3  soup.yaml
175f3ad1ae66034c557f7d54efe30121c6e5b7e99169127539dacb2207e3df67  data/train.jsonl
ae130e18936df26dbd9db8921f2c131bff394236cb620e7ec2c12ed834c6b4c2  data/eval.jsonl
981595fe71f26feb321efe2bf9af8cdbc76365141daeb1f52979c85920b38553  prepare_data.py
f44ce1a3e7f52e280bb1f9add56c15597fe99d46bba008ec8da932a63d89b822  verify_training.py
dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee  /content/models/qwen2.5-1.5b-pinned/model.safetensors

=== GPU BEFORE ===
Thu Oct  1 08:04:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+---------------

In [13]:
%%bash
cd /content

echo "=== INVALIDATING / FALLBACK SIGNALS ==="
grep -n -iE \
"traceback|exception|skipp|retry|reduc|halv|auto_find_batch|batch size|overflow|nan|inf\b|meta" \
logs/train_final_v3/training.log || true

echo
echo "=== CONTEXT AROUND CUDA ALLOCATION WARNINGS ==="
grep -n -B3 -A3 -iE "allocation failed|OOM" \
logs/train_final_v3/training.log || true

echo
echo "=== SAVED ADAPTER ==="
ls -lah output/dpo_qwen15b_stream

echo
echo "=== ADAPTER TENSOR CHECK ==="
/content/venv312/bin/python - <<'PY'
from safetensors.torch import load_file
from pathlib import Path

p = Path("output/dpo_qwen15b_stream/adapter_model.safetensors")
assert p.exists(), f"missing: {p}"

sd = load_file(str(p))
b = [k for k in sd if "lora_B" in k]

nonzero = sum(bool(sd[k].abs().sum().item() > 0) for k in b)
inner = sum(".inner." in k for k in sd)
meta_like = [k for k, v in sd.items() if str(v.device) == "meta"]

print("total keys:", len(sd))
print("lora_B tensors:", len(b))
print("nonzero lora_B:", nonzero)
print("zero lora_B:", len(b) - nonzero)
print(".inner. keys:", inner)
print("meta tensors:", len(meta_like))
PY

echo
echo "=== RAW NVIDIA-SMI PEAK ==="
/content/venv312/bin/python - <<'PY'
import csv

vals = []
with open("logs/train_final_v3/nvidia_smi_sampling.csv", newline="") as f:
    for row in csv.reader(f):
        if not row:
            continue
        try:
            # timestamp, name, memory.used, memory.total, utilization.gpu
            vals.append(int(row[2].strip().split()[0]))
        except (ValueError, IndexError):
            pass

print("samples:", len(vals))
print("peak MiB:", max(vals))
print("peak GiB:", round(max(vals)/1024, 3))
PY

=== INVALIDATING / FALLBACK SIGNALS ===
56:[transformers] The model is already on multiple devices. Skipping the move to device specified in `args`.

=== CONTEXT AROUND CUDA ALLOCATION WARNINGS ===
58-Training started!
59-
60-[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 63/63 [09:49<00:00,  9.35s/it]
67-{'loss': '0.6899', 'grad_norm': '5.871', 'learning_rate': '4.844e-05', 'entropy': '1.556', 'num_tokens': '4.863e+04', 'logits/chosen': '-2.621', 'logits/rejected': '-2.586', 'mean_token_accuracy': '0.5905', 'rewards/chosen': '0.03305', 'rewards/rejected': '0.02593', 'rewards/accuracies': '0.3375', 'rewards/margins': '0.007128', 'logps/chosen': '-214.4', 'logps/rejected': '-184.4', 'epoch': '0.16'}
68-{'loss': '0.6888', 'grad_norm': '4

In [14]:
%%bash
cd /content

mkdir -p logs/verify

(date -u; \
 /content/venv312/bin/python verify_training.py \
   --adapter output/dpo_qwen15b_stream) \
  2>&1 | tee logs/verify/post_final.log

echo
echo "=== ADAPTER SHA256 ==="
sha256sum output/dpo_qwen15b_stream/adapter_model.safetensors

Thu Oct  1 08:22:27 AM UTC 2026
Loading weights: 100%|██████████| 338/338 [00:12<00:00, 26.60it/s]
[post] {"n": 50, "mean_logp_chosen": -231.7264, "mean_logp_rejected": -247.9247, "mean_margin": 16.1983, "median_margin": -6.4651, "frac_chosen_gt_rejected": 0.46, "mean_margin_per_tok": 0.1766}
[params] file_keys=392 inner_keys=0 lora_B=196 lora_B_nonzero=196 loaded_and_equal=392/392 lora_B_norm[min/median/max]=1.683e-02/4.467e-02/1.091e-01
[post vs baseline] mean_delta=0.2217 ci95=[-0.4218544692993164, 0.8898627529144286] frac_pos=0.54 frac_neg=0.46 sign_p=0.336 corr(Δ,len_diff)=0.49906912310109175 -> PARAMS_CHANGED_ONLY (outputs changed, no held-out preference shift)
wrote artifacts/post_logps.json
[control_random_adapter] mean_delta=-0.0107 ci95=[-0.05096636867523193, 0.033551553726196275] -> PARAMS_CHANGED_ONLY (outputs changed, no held-out preference shift)
[control_random_adapter] EXPECTED: not TRAINED -> PASS
wrote artifacts/control_random_adapter.json

=== ADAPTER SHA256 ===
7806

In [15]:
!/content/venv312/bin/soup ship --help

                                                                                
 Usage: soup ship [OPTIONS] [COMMAND] [ARGS]...                                 
                                                                                
 SHIP / DON'T SHIP verdict after fine-tuning: task win AND no catastrophic      
 forgetting, fused into one decision (v0.71.25).                                
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --base                          TEXT     Base model id/path (the 'before').  │
│ --tuned                         TEXT     Tuned model id/path (a separate     │
│                                          'after' model).                     │
│ --adapter                       TEXT     LoRA adapter path (tuned = base +   │
│                                          this adapter).                      │
│ --task-eval               

In [16]:
%%bash
cd /content
mkdir -p artifacts logs/ship

/content/venv312/bin/python - <<'PY'
import json

src = "data/eval.jsonl"
dst = "data/ship_tasks.jsonl"

with open(src, encoding="utf-8") as f, open(dst, "w", encoding="utf-8") as out:
    n = 0
    for line in f:
        r = json.loads(line)
        task = {
            "prompt": r["prompt"],
            "expected": r["chosen"],
            "scoring": "semantic",
            "category": "russian_preference_proxy",
        }
        out.write(json.dumps(task, ensure_ascii=False) + "\n")
        n += 1

print("ship tasks:", n)
assert n == 50
PY

sha256sum data/ship_tasks.jsonl
head -1 data/ship_tasks.jsonl

ship tasks: 50
161ba8f38ad509384e4d4ec0a2fd7f10ee749ba1a9846a6fcdd7e64ffb589e46  data/ship_tasks.jsonl
{"prompt": "Человек: что мне делать, если я кашляю кровью?\n\nПомощник: Если вы кашляете кровью, это признак более серьезного заболевания. Как и во всех медицинских ситуациях, вы должны обратиться к врачу, чтобы получить совет от медицинского работника. Есть ли у вас медицинская страховка?\n\nЧеловек: Нет, не знаю.\n\nАссистент:", "expected": "К сожалению, я боюсь, что без медицинской страховки вы не всегда можете позволить себе врача. Часто можно получить медицинские услуги со скидкой в ​​Соединенных Штатах, если вы можете доказать, что у вас очень ограниченный доход, или если вы являетесь ветераном вооруженных сил. В противном случае, если вы не застрахованы, возможно, ваша медицинская страховка была отклонена медицинской страховой компанией, и вам придется заплатить за посещение из собственного кармана. Также возможно, что вы не застрахованы, и это может быть состояние здоровья, ко

In [17]:
%%bash
cd /content
set +e

/content/venv312/bin/soup ship \
  --base /content/models/qwen2.5-1.5b-pinned \
  --adapter output/dpo_qwen15b_stream \
  --task-eval data/ship_tasks.jsonl \
  --task-mode metric \
  --device cuda \
  --config soup.yaml \
  --output artifacts/soup_ship_verdict.json \
  --emit-evidence artifacts/soup_ship_evidence.json \
  2>&1 | tee logs/ship/soup_ship_final.log

STATUS=${PIPESTATUS[0]}

echo
echo "=== SOUP SHIP EXIT CODE ==="
echo "$STATUS"

echo
echo "=== VERDICT JSON ==="
cat artifacts/soup_ship_verdict.json 2>/dev/null || true

Live eval: loading base/tuned at full precision (bfloat16); pass --config to 
reuse the training run's own quantization.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1067.64it/s]
╭───────────────────────────────── soup ship ──────────────────────────────────╮
│ DON'T SHIP                                                                   │
│ Leg 1 task win (metric): 0.0000 -> 0.0000  [no win]                          │
│                                                                              │
│ Leg 2 general suite (threshold 5.00%)                                        │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓                  │
│ ┃ Benchmark         ┃   Base ┃  Tuned ┃       Δ ┃ Verdict ┃                  │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩                  │
│ │ mini_arithmetic   │ 0.9722 │ 1.0000 │ +0.0278 │ ok      │                  │
│ │ mini_common_

In [18]:
%%bash
cd /content

rm -f final_submission_bundle.zip

zip -r final_submission_bundle.zip \
  soup.yaml \
  prepare_data.py \
  verify_training.py \
  run_final.sh \
  materialize_model.py \
  data/train.jsonl \
  data/eval.jsonl \
  data/ship_tasks.jsonl \
  logs \
  artifacts \
  output/dpo_qwen15b_stream/adapter_config.json \
  output/dpo_qwen15b_stream/adapter_model.safetensors \
  output/dpo_qwen15b_stream/README.md \
  output/dpo_qwen15b_stream/tokenizer_config.json \
  output/dpo_qwen15b_stream/tokenizer.json \
  2>/dev/null

echo
echo "=== BUNDLE ==="
ls -lh final_submission_bundle.zip
sha256sum final_submission_bundle.zip

  adding: soup.yaml (deflated 41%)
  adding: prepare_data.py (deflated 72%)
  adding: verify_training.py (deflated 63%)
  adding: run_final.sh (deflated 55%)
  adding: materialize_model.py (deflated 44%)
  adding: data/train.jsonl (deflated 75%)
  adding: data/eval.jsonl (deflated 74%)
  adding: data/ship_tasks.jsonl (deflated 74%)
  adding: logs/ (stored 0%)
  adding: logs/train_final_v3/ (stored 0%)
  adding: logs/train_final_v3/run_info.txt (deflated 33%)
  adding: logs/train_final_v3/nvidia_smi_before.txt (deflated 76%)
  adding: logs/train_final_v3/training.log (deflated 76%)
  adding: logs/train_final_v3/nvidia_smi_sampling.csv (deflated 89%)
  adding: logs/train_final_v3/nvidia_smi_after.txt (deflated 77%)
  adding: logs/train_final_v3_aborted_prompt/ (stored 0%)
  adding: logs/train_final_v3_aborted_prompt/run_info.txt (deflated 33%)
  adding: logs/train_final_v3_aborted_prompt/nvidia_smi_before.txt (deflated 77%)
  adding: logs/train_final_v3_aborted_prompt/training.log (defla

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
!cp /content/final_submission_bundle.zip /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/final_submission_bundle.zip

-rw------- 1 root root 68M Oct  1 09:49 /content/drive/MyDrive/final_submission_bundle.zip
